# Archive notice
> This is the original 2025 team notebook, retained for provenance. Its advanced-model section used same-match statistics and a `HighScoring` feature derived from final goals, so the reported 54-56% figures are not valid pre-match forecasts. Use `src/portfolio_pipeline.py` for the corrected, chronological, pre-match-only evaluation.


# **CIS 5450 Final Project - Football Match Result Prediction**
*Lucas Qu, Leo Lin, & Hongru Da*



# Part 1: Introduction




For our final project, we sought to understand what drives the outcome of professional soccer matches; specifically, whether the home team wins, the match ends in a draw, or the away team wins. Using over 25 years of historical match data from English leagues (50,000+ matches), we created a unified cleaned dataframe by standardizing dates, numeric columns, match statistics, and bookmaker odds.

With this dataset, we conducted exploratory data analysis (EDA) to examine the distribution of match outcomes, trends in home-field advantage and goal scoring over time, and the relationship between bookmaker implied probabilities and actual results. We also analyzed correlations among numeric features (goals, shots, corners, cards) and assessed missingness to determine which features to retain. These insights informed our feature selection and confirmed that odds and match statistics provide meaningful predictive signals.

We frame match outcome prediction as a multi-class classification problem, using the full-time result (FTR) as the target. Our baseline models include always predicting a home win, predicting the bookmaker favorite, and logistic regression using only odds features. We then compare these to more flexible models such as Naïve Bayes and tree-based methods (Random Forest and Gradient Boosted Trees) that can capture nonlinear interactions. Performance is evaluated using accuracy and log-loss.

Throughout our notebook, we pair each visualization and model with a brief explanation of its motivation and key takeaways. Our goal is to highlight which factors including team form, match statistics, and bookmaker pricing most influence match outcomes. We look forward to sharing our findings in more detail.

# Part 2: Data Processing & Preloading


For this step, we imported the core libraries required for data loading, preprocessing, exploratory analysis, and baseline modeling. Our analysis relies primarily on pandas and numpy for data manipulation, seaborn and matplotlib for visualization, scikit-learn for implementing our baseline classification models, and requests for downloading match data directly from online sources.

In [ ]:
# Imports and Installs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import duckdb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, log_loss
)
import folium
from folium.plugins import HeatMap
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

## 2.1 Loading & Preprocessing Match Data

### 2.1.1 Loading Data & Converting to Dataframe



Given that **[England Football Results](https://Football-Data.co.uk)** provides each season and league as an individual CSV file, our first step was to download multiple seasons of match results across all major English divisions. We then combined these files into a single dataframe for further cleaning. Because each file is only a few megabytes, standard pandas functionality is sufficient for loading and processing the data.

Not all season–league combinations are available on the website, particularly for older years or lower divisions. As a result, some download attempts were skipped when a corresponding CSV file did not exist on the server. This is expected behavior due to incomplete historical coverage across leagues. After loading all available files, we concatenated them into one unified dataset containing more than 50,000 matches spanning over 25 years of English football.

In [ ]:
# Seasons available
seasons = [
    "2324","2223","2122","2021","1920","1819","1718","1617","1516","1415",
    "1314","1213","1112","1011","0910","0809","0708","0607","0506","0405",
    "0304","0203","0102","0001","9900","9899"
]

# All English leagues (Football-Data codes)
leagues = ["E0","E1","E2","E3","E4","E5","E6","E7","EC"]

df_list = []

for season in seasons:
    for league in leagues:
        url = f"https://www.football-data.co.uk/mmz4281/{season}/{league}.csv"

        try:
            df = pd.read_csv(url, encoding="latin-1")
            df["Season"] = season
            df["League"] = league
            df_list.append(df)
            print(f"Loaded: {url}")
        except:
            print(f"Skipping unavailable file: {url}")


In [ ]:
# Combine all seasons into a single dataframe
soccer_df = pd.concat(df_list, ignore_index=True)
display(len(soccer_df))
display(soccer_df['Season'].unique())

soccer_df_original = soccer_df

soccer_df

We also decided to incorporate locational data that was not already encoded in the dataset itself. This is simple to do via a merge on the team names with an external dataset with the location data.

In [ ]:
stadiums = pd.read_csv("http://opisthokonta.net/wp-content/uploads/2015/03/stadiums_20150302.csv")

# Step 1: Give unique names so we can safely drop
stadiums = stadiums.rename(columns={
    "FDCOUK": "TeamShort",
    "Team": "TeamLong"
})

# Step 2: Drop the long team names
stadiums = stadiums.drop(columns=["TeamLong"])

# Step 3: Rename short name to Team
stadiums = stadiums.rename(columns={"TeamShort": "Team"})

# Step 4: Keep only location columns
stadiums_base = stadiums[["Team", "City", "Stadium",
                          "Capacity", "Latitude", "Longitude", "Country"]]

stadiums_base





Before merging the match dataset with the stadium/location table, we used regex to standardize team names by removing punctuation and collapsing extra spaces. This ensures that team identifiers match across both datasets and prevents join failures caused by inconsistent text formatting.

In [ ]:
# Clean HomeTeam and AwayTeam before merging with stadiums table

soccer_df["HomeTeam"] = (
    soccer_df["HomeTeam"]
    .str.replace(r"[^A-Za-z0-9 ]+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

soccer_df["AwayTeam"] = (
    soccer_df["AwayTeam"]
    .str.replace(r"[^A-Za-z0-9 ]+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [ ]:
# Home location
home_stadiums = stadiums_base.add_prefix("Home_")

soccer_df = soccer_df.merge(
    home_stadiums,
    left_on="HomeTeam",
    right_on="Home_Team",
    how="left",
).drop(columns=["Home_Team"])

# Away location
away_stadiums = stadiums_base.add_prefix("Away_")

soccer_df = soccer_df.merge(
    away_stadiums,
    left_on="AwayTeam",
    right_on="Away_Team",
    how="left",
).drop(columns=["Away_Team"])




In addition to match statistics and betting-odds features, we introduce **TravelDistance_km** as a geographic feature to capture the effect of away-team travel burden. Travel distance is computed using the **Haversine formula**, which calculates the great-circle distance between two latitude–longitude points on the Earth’s surface:

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

soccer_df["TravelDistance_km"] = soccer_df.apply(
    lambda row: haversine(
        row["Home_Latitude"], row["Home_Longitude"],
        row["Away_Latitude"], row["Away_Longitude"],
    ),
    axis=1,
)


### 2.1.2 Analyzing Data Structure

To better understand the structure of our combined match dataset, we began by examining summary statistics across key numerical features such as goals scored, shots, fouls, cards, and betting odds. This allows us to identify the overall distribution of values, potential outliers, and differences in scale across variables. Understanding these characteristics early on helps inform preprocessing choices and ensures that later EDA and modeling steps are based on a clear understanding of the underlying data.



In [ ]:
# Descriptive statistics for numeric match features
soccer_df.describe()

The descriptive statistics from our combined dataset revealed several important patterns:


1.   **Match outcomes and scoring behavior are highly variable across leagues.**

The mean number of goals scored by the home and away teams is relatively close, but the maximum values highlight occasional high-scoring matches. The spread in goals, cards, and shots suggests that English football exhibits substantial match-to-match volatility, which can influence prediction difficulty.

2.   **Betting odds span a wide numerical range and differ significantly in scale**

Home-win, draw, and away-win odds can vary from very low values (heavy favorites) to extremely high values (major underdogs). This wide range reinforces the importance of later normalizing or scaling these variables before training models such as logistic regression.

3.  **Some statistical categories contain partial missingness across seasons.**

Features such as shots, corners, and fouls are not consistently recorded in older seasons or lower-division leagues. This uneven feature availability must be considered when deciding whether to impute values, remove certain columns, or limit analysis to seasons with full data coverage.

Together, these observations give us a clearer picture of the structure and variability within our dataset and help inform the preprocessing steps and feature selections that follow.

In [ ]:
# Dataframe column datatypes
soccer_df.dtypes

In [ ]:
# First 10 rows of the combined dataset
soccer_df.head(10)

After inspecting the first few rows of our combined dataset, we noticed that several key categorical columns contain repeated values across different seasons and leagues. To better understand the structure of these features, we examined the unique values present in columns such as `League`, `Season`, and `FTR` (full-time result).

**League**: identifies the division in which the match was played.

E0: Premier League (Tier 1)

E1: Championship (Tier 2)

E2: League One (Tier 3)

E3: League Two (Tier 4)

EC: National League / Conference (varies by season)

**Season**: encoded as a four-digit shorthand indicating the two years spanned by the season (e.g., 2324 → 2023–2024, 1617 → 2016–2017).

**FTR** (Full-Time Result): the match outcome.

H — Home Win

D — Draw

A — Away Win

Identifying the categorical makeup of these variables helps us confirm that our data loading process was successful and provides insight into how matches are distributed across divisions and years. It also ensures that our target variable (FTR) is correctly encoded and ready for subsequent modeling steps.

In [ ]:
# Unique divisions present in the dataset
soccer_df['League'].unique()

In [ ]:
# Unique seasons included in the dataset
soccer_df['Season'].unique()

In [ ]:
# Unique match outcomes:
soccer_df['FTR'].unique()


The distinct values in these columns confirm that our dataset successfully aggregates matches from multiple divisions and seasons. The presence of several league codes indicates that we captured a broad range of competitive levels across English football. Similarly, the range of seasons validates that our web-scraped data spans over two decades. Finally, inspecting the unique match outcomes helps ensure that the target variable is correctly encoded and ready for later modeling.

### 2.1.3 Dropping Insignificant Columns & Handling Null Rows

In the Football-Data dataset, several columns either contain identifiers that are not informative for prediction or contain a substantial number of missing values. To simplify the dataset and avoid introducing noise into our models, we removed columns that do not contribute meaningful predictive power.

**Dropping identifier columns**

Columns such as `Div`, `Referee`, `HomeTeam`, and `AwayTeam` contain names or identifiers that are not useful for modeling match outcomes directly. Team names are better incorporated later through engineered features (e.g., recent form), and referee names have no strong evidence of predictive value for our classification task.


In [ ]:
# Number of rows where the full-time result (FTR) is missing
cols_to_drop = ["Referee", "Div"]  # keep HomeTeam/AwayTeam for later feature engineering
soccer_df = soccer_df.drop(columns=cols_to_drop, errors="ignore")


**Dropping rows with missing match outcomes (FTR)**

The `FTR` column is the target variable (H, D, A).
Rows where `FTR` is missing represent incomplete or future fixtures and cannot be used for supervised learning.

In [ ]:
# Dropping all rows where the match outcome (FTR) is missing
soccer_df = soccer_df.dropna(subset=["FTR"])

display(len(soccer_df))

soccer_df.head(10)

**Dropping columns with excessive missingness**

Some statistical features (like shots, corners, fouls) are inconsistently recorded in older seasons and lower-division leagues. Columns where the majority of rows are missing will not contribute meaningfully to the model.

We drop any column with more than 40% missing values:

In [ ]:
# Checking percentage of missing values for each column
missing_ratio = soccer_df.isna().mean()


In [ ]:
# Dropping columns with >40% missing values
cols_to_remove = missing_ratio[missing_ratio > 0.40].index
soccer_df = soccer_df.drop(columns=cols_to_remove)

**Dropping columns for the Extended Betting Market**

To streamline the dataset, we removed several bookmaker columns associated with extended betting markets such as Asian Handicap and alternative totals. These odds are inconsistently reported across seasons and are not directly relevant to predicting the basic full-time result (Home/Draw/Away). Dropping them reduces missingness and keeps our analysis focused on the core 1X2 odds most important for our prediction task.

In [ ]:
cols_to_drop = ['Bb1X2','BbMxH','BbAvH','BbMxD','BbAvD','BbMxA','BbAvA',
                'BbOU','BbMx>2.5','BbAv>2.5','BbMx<2.5','BbAv<2.5',
                'BbAH','BbAHh','BbMxAHH','BbAvAHH','BbMxAHA','BbAvAHA',
                'LBH','LBD','LBA']

soccer_df = soccer_df.drop(columns=cols_to_drop)

**Dropping invalid betting odds**

In [ ]:
b365_cols = ['B365H', 'B365D', 'B365A']

# Find rows where any Bet365 odds are <= 0
invalid_b365 = soccer_df[(soccer_df[b365_cols] <= 0).any(axis=1)]

print("Number of rows with invalid B365 odds:", len(invalid_b365))
invalid_b365.head()


soccer_df = soccer_df[(soccer_df[b365_cols] > 0).all(axis=1)]

### 2.1.4 Removing Outliers & Impossible Values
To prepare the dataset for EDA and modeling, we removed inconsistent or impossible game records.

**Removing matches with impossible scorelines**

Some older datasets contain errors such as negative goals or extremely high values that do not reflect real matches.

In [ ]:
# Number of matches with negative goals (should be zero)
((soccer_df['FTHG'] < 0) | (soccer_df['FTAG'] < 0)).sum()


**Removing irrelevant betting markets**

Lots of these betting markets simply do not supply odds in certain years. They are also highly correlated with each other, so we simply keep the Bet365 odds columns and drop the rest.

In [ ]:
# Dropping matches with negative goal values
soccer_df = soccer_df[(soccer_df['FTHG'] >= 0) & (soccer_df['FTAG'] >= 0)]

soccer_df.head()


In [ ]:
cols_to_drop = [
    'BWH', 'BWD', 'BWA',
    'IWH', 'IWD', 'IWA',
    'WHH', 'WHD', 'WHA',
    'VCH', 'VCD', 'VCA'
]

soccer_df = soccer_df.drop(columns=cols_to_drop, errors='ignore')


Now that our dataset is cleaned, we have removed irrelevant identifiers, rows without known match outcomes, and columns with extensive missing values. We also filtered out impossible scorelines and invalid betting odds to ensure that the remaining match records are realistic and usable. These preprocessing steps improve the reliability of our exploratory analysis and downstream modeling performance.

To validate that our cleaning pipeline did not accidentally remove entire seasons, leagues, or distort match distributions, we used DuckDB to run SQL-style summary queries directly on the cleaned soccer_df. By aggregating matches by season and league and examining average goals scored, we were able to confirm that data coverage remained intact after all filtering and merging operations.


In [ ]:
# Use DuckDB to run a SQL sanity check on the cleaned dataframe
season_league_summary = duckdb.query("""
    SELECT
        Season,
        League,
        COUNT(*) AS n_matches,
        AVG(FTHG) AS avg_home_goals,
        AVG(FTAG) AS avg_away_goals
    FROM soccer_df
    GROUP BY Season, League
    ORDER BY Season, League
""").to_df()

season_league_summary.head()

After completing these preprocessing and validation steps, we defined the subset of features most relevant for predicting match outcomes. We retained variables such as the match date, league, season, final score (FTHG/FTAG), and core bookmaker odds (e.g., B365H, B365D, B365A), as well as consistently reported match statistics. We excluded high-missingness features and other identifiers that do not provide predictive signal. Establishing this final feature set ensures that our subsequent EDA and modeling efforts operate on clean, reliable, and meaningful inputs.

# Part 3: Exploratory Data Analysis

## 3.1 Effects of Location on Win Rate

### 3.1.1 Concentration of Football Matches in England

To better understand the geographic distribution of match locations, we generated a heatmap using each home team’s stadium coordinates. This visualization allowed us to observe where matches were most heavily concentrated across the league. As expected, the strongest cluster appeared in London, reflecting the city’s high density of clubs and the large number of fixtures hosted there each season. Additional clusters emerged in the Midlands and northern England, regions with historically active football hubs such as Birmingham, Manchester, Liverpool, Leeds, and Newcastle. By mapping these locations, we not only verified the accuracy of our stadium data but also gained a clearer picture of the league’s geographic footprint and the natural concentration of matches across the country.

In [ ]:
# Keep only rows with valid, non-zero coordinates

valid = (
    soccer_df["Home_Latitude"].notna() &
    soccer_df["Home_Longitude"].notna() &
    (soccer_df["Home_Latitude"] != 0) &
    (soccer_df["Home_Longitude"] != 0)
)

valid = soccer_df[valid]
print(valid[["HomeTeam", "Home_Latitude", "Home_Longitude"]].head())

# Use only cleaned home coords for now
heat_data = valid[["Home_Latitude", "Home_Longitude"]].values.tolist()

# Center explicitly over the UK instead of using the mean
m = folium.Map(location=[54.5, -2.0], zoom_start=6)   # roughly centre of the UK

HeatMap(
    heat_data,
    radius=12,
    blur=18,
    max_zoom=8
).add_to(m)

# Display the map
m


In [ ]:
# Aggregate matches + home wins per stadium
stadium_stats = (
    soccer_df
    .groupby(["HomeTeam", "Home_Stadium"], dropna=True)
    .agg(
        matches=("FTR", "size"),
        home_wins=("FTR", lambda x: (x == "H").sum())
    )
    .reset_index()
)

# Sort by number of home wins
top_wins = stadium_stats.sort_values("home_wins", ascending=False).head(15)

stadium_stats["home_win_rate"] = stadium_stats["home_wins"] / stadium_stats["matches"]

# Filter to stadiums with enough matches to be meaningful
min_matches = 50
eligible = stadium_stats[stadium_stats["matches"] >= min_matches]

# Top 15 stadiums by home win rate
top_rates = eligible.sort_values("home_win_rate", ascending=False).head(15)

plt.figure(figsize=(12, 6))
plt.bar(top_rates["Home_Stadium"], top_rates["home_win_rate"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Home Win Rate")
plt.title(f"Top 15 Stadiums by Home Win Rate (≥ {min_matches} Home Matches)")
plt.ylim(0, 1)  # since it's a rate
plt.tight_layout()
plt.show()



In [ ]:
# ---- attach coordinates to the top 15 by win rate ----
stadium_coords = (
    soccer_df[["Home_Stadium", "Home_Latitude", "Home_Longitude"]]
    .drop_duplicates()
)

top_rates_map = top_rates.merge(stadium_coords, on="Home_Stadium", how="left")

# keep only valid coordinates
top_rates_map = top_rates_map[
    top_rates_map["Home_Latitude"].notna() &
    top_rates_map["Home_Longitude"].notna() &
    (top_rates_map["Home_Latitude"] != 0) &
    (top_rates_map["Home_Longitude"] != 0)
]

# ---- helper for color based on win rate ----
def winrate_color(w):
    if w < 0.4:
        return "red"
    elif w < 0.6:
        return "orange"
    elif w < 0.7:
        return "gold"
    else:
        return "green"

# ---- make the map ----
m = folium.Map(location=[54.5, -2.0], zoom_start=6)   # centre over UK

max_matches = top_rates_map["matches"].max()

for _, row in top_rates_map.iterrows():
    wr = row["home_win_rate"]
    color = winrate_color(wr)
    radius = 5 + (row["matches"] / max_matches) * 20  # bigger bubble = more matches

    folium.CircleMarker(
        location=[row["Home_Latitude"], row["Home_Longitude"]],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=(
            f"<b>{row['Home_Stadium']}</b><br>"
            f"Home Win Rate: {wr*100:.1f}%<br>"
            f"Matches: {row['matches']}"
        ),
        tooltip=f"{row['Home_Stadium']} – {wr*100:.1f}% ({row['matches']} matches)"
    ).add_to(m)

m

## 3.2 Match Results & Goal Statistics

### 3.2.1 Distribution of Full-Time Results (FTR)

We begin this section by examining the distribution of full-time match results (Home Win, Draw, Away Win). Understanding how frequently each outcome occurs provides an essential baseline for both EDA and predictive modeling, as the match result is our primary target variable. This distribution reveals overall patterns of competitiveness, highlights the presence of home-field advantage, and helps identify class imbalance that may influence model training. By comparing the proportions of each result, we gain initial insight into structural tendencies across matches.


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='FTR', data=soccer_df, order=['H','D','A'], palette='Set2')

plt.title("Distribution of Full-Time Match Results")
plt.xlabel("Match Result (H=Home Win, D=Draw, A=Away Win)")
plt.ylabel("Count")
plt.show()

soccer_df.head()

Takeaways:

* Home wins occur most frequently, highlighting the presence of a strong home-field advantage across leagues.

* Draws are less common but still represent a significant share of match outcomes, indicating that many matches remain competitive and evenly matched.

* Away wins are the least frequent result, suggesting that travel, crowd influence, and pitch familiarity contribute meaningfully to team performance.

* The distribution reveals noticeable class imbalance, which should be addressed in any predictive modeling pipeline (e.g., stratified sampling or class weighting).

* Comparing distributions across seasons or leagues may uncover shifts in competitiveness or changes in home-field dynamics.


### 3.2.2 Goals Distribution

Next, we analyze goal-scoring patterns across matches. Goal distributions for both home and away teams help characterize the offensive and defensive tendencies within the dataset. Examining histograms of full-time goals (FTHG, FTAG) and total match goals allows us to understand how frequently high-scoring or low-scoring matches occur. These insights are valuable when designing models that rely on score-related features, and they also provide useful context for understanding betting markets and common outcome expectations.


In [ ]:
soccer_df['TotalGoals'] = soccer_df['FTHG'] + soccer_df['FTAG']

fig, axs = plt.subplots(1, 3, figsize=(16,5))

sns.histplot(soccer_df['FTHG'], kde=True, ax=axs[0], color='blue')
axs[0].set_title("Distribution of Home Goals")

sns.histplot(soccer_df['FTAG'], kde=True, ax=axs[1], color='red')
axs[1].set_title("Distribution of Away Goals")

sns.histplot(soccer_df['TotalGoals'], kde=True, ax=axs[2], color='green')
axs[2].set_title("Distribution of Total Match Goals")

plt.tight_layout()
plt.show()

soccer_df.head(10)

Takeaways:

* Goal distributions for both home and away teams are right-skewed, with the vast majority of matches producing 0–2 goals per side.

* Home teams score slightly more on average, reinforcing the earlier observation of a measurable home-field advantage.

* Total match goals peak at 2–3, aligning with standard betting market totals (e.g., Over/Under 2.5).

* High-scoring matches (4+ goals) are rare but still meaningful, as they may indicate structural mismatches between teams or tactical styles.

* These distributions are consistent with low-scoring sports, which explains why small differences in expected goals can heavily influence odds.

* Understanding goal distributions helps model calibration, especially if you later predict probabilities or expected goal counts.

### 3.2.3 Home vs. Away Goal Advantage

Home-field advantage is a well-documented phenomenon in football, and analyzing goal differences between home and away teams provides a quantitative view of this effect. By comparing the distributions of FTHG and FTAG, we can assess how consistently teams perform better at home. This comparison highlights structural imbalances in scoring and can inform feature engineering, baseline models, and downstream predictive tasks such as outcome prediction or expected-goals modeling.


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(data=soccer_df[['FTHG','FTAG']], palette=['blue','red'])
plt.title("Home vs Away Goal Distribution")
plt.ylabel("Goals")
plt.xticks([0,1], ['Home Goals', 'Away Goals'])
plt.show()

soccer_df.head(10)


Takeaways:

* Home teams most commonly score 1–2 goals, with relatively few 0-goal matches, reflecting stronger offensive performance at home.

* Away teams are tightly concentrated around 0–2 goals, with a noticeably higher frequency of scoreless matches, indicating more conservative or less effective away play.

* The central distributions of home and away goals are similar, but home teams show slightly higher scoring and a few more high-end outliers.

* These scoring differences reinforce the structural home-field advantage and justify including features that capture home-vs-away effects in predictive models.

### 3.2.4 First-Half vs Second-Half Goal Behavior

Football matches often unfold differently between the first and second halves, influenced by tactical adjustments, substitutions, and player fatigue. By examining first-half goals (HTHG, HTAG) alongside full-time goals, we can identify how scoring intensity changes as the match progresses. This analysis helps reveal whether matches generally open up in the second half and provides insight into in-game dynamics that could be valuable for modeling momentum, live prediction, or temporal scoring patterns.


In [ ]:
soccer_df['1stHalfGoals'] = soccer_df['HTHG'] + soccer_df['HTAG']
soccer_df['2ndHalfGoals'] = soccer_df['TotalGoals'] - soccer_df['1stHalfGoals']

plt.figure(figsize=(8,5))
sns.histplot(soccer_df['1stHalfGoals'], kde=True, color='purple', label='1st Half Goals')
sns.histplot(soccer_df['2ndHalfGoals'], kde=True, color='orange', label='2nd Half Goals')

plt.title("First Half vs Second Half Goals")
plt.xlabel("Goals")
plt.legend()
plt.show()

soccer_df.head()


Takeaways:

* Both halves peak at 1 goal, but the shape of the distributions differs substantially.

* First halves contain far more 0-goal periods, showing that teams often start conservatively or defensively.

* Second-half distributions have many more 2-, 3-, 4-goal outcomes, meaning matches tend to open up and produce more scoring as they progress.

* This late-scoring pattern is useful for live-prediction models, expected-goals adjustments, and feature engineering involving momentum or time-dependent scoring.

* Differences in first- vs second-half scoring highlight the importance of modeling match context — especially whether the halftime score influences second-half behavior.

## 3.3 Team-Level Performance Patterns

### 3.3.1 Number of Shots vs Goals

Understanding the relationship between a team’s shot volume and its goal-scoring output is fundamental to evaluating offensive effectiveness. Not all shots are created equal; shots on target tend to be far more predictive of scoring, while overall shot totals reflect general attacking presence. In this subsection, we explore how home and away shot totals relate to goals scored, and whether accuracy (HST/AST) plays a stronger predictive role than raw shot volume (HS/AS). A correlation heatmap helps summarize how these variables interact.

In [ ]:
plt.figure(figsize=(16,5))

# Scatter: Home Shots vs Home Goals
plt.subplot(1,3,1)
sns.scatterplot(data=soccer_df, x='HS', y='FTHG', alpha=0.5, color='blue')
plt.title("Home Shots vs Home Goals")
plt.xlabel("Home Shots (HS)")
plt.ylabel("Home Goals (FTHG)")

# Scatter: Away Shots vs Away Goals
plt.subplot(1,3,2)
sns.scatterplot(data=soccer_df, x='AS', y='FTAG', alpha=0.5, color='red')
plt.title("Away Shots vs Away Goals")
plt.xlabel("Away Shots (AS)")
plt.ylabel("Away Goals (FTAG)")

# Correlation heatmap
plt.subplot(1,3,3)
corr_vars = soccer_df[['HS', 'HST', 'FTHG', 'AS', 'AST', 'FTAG']]
sns.heatmap(corr_vars.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation: Shots, Shots on Target, Goals")

plt.tight_layout()
plt.show()

soccer_df.head()

Takeaways:

* Total shots (HS/AS) show only a weak relationship with goals scored, meaning that simply taking many shots does not guarantee scoring success.

* Shots on target (HST/AST) correlate more strongly with goals than total shots do, confirming that shot quality matters more than shot quantity, but even these correlations are only moderate.

* HS and HST are strongly correlated (0.65), reflecting that teams producing more shots naturally generate more shots on target. Because of this, including both variables in predictive models may introduce multicollinearity.

* The scatterplots show large spreads, with many matches producing many shots but few goals, consistent with blocked shots, poor finishing, or strong goalkeeper performance.

* Overall, shots on target are a more informative feature for predictive modeling than total shots, but neither metric alone provides strong predictive power. They should be combined with additional context such as expected goals (xG), possession, and team strength.

### 3.3.2 Fouls vs Cards

This subsection examines team discipline by analyzing how fouls committed relate to yellow and red cards received. Fouls (HF/AF) represent a team's physical or aggressive play style, while card accumulation (HY/HY, HR/AR) reflects referee decisions and the severity of infractions. Understanding this relationship helps identify whether certain teams or match contexts lead to more punitive outcomes and whether fouls reliably escalate into bookings.

In [ ]:
plt.figure(figsize=(14,5))

# Home fouls vs yellow cards
plt.subplot(1,2,1)
sns.scatterplot(data=soccer_df, x='HF', y='HY', alpha=0.5, color='green')
plt.title("Home Fouls vs Home Yellow Cards")
plt.xlabel("Home Fouls (HF)")
plt.ylabel("Home Yellow Cards (HY)")

# Away fouls vs yellow cards
plt.subplot(1,2,2)
sns.scatterplot(data=soccer_df, x='AF', y='AY', alpha=0.5, color='orange')
plt.title("Away Fouls vs Away Yellow Cards")
plt.xlabel("Away Fouls (AF)")
plt.ylabel("Away Yellow Cards (AY)")

plt.tight_layout()
plt.show()

soccer_df.head()

Takeaways:

* A very weak, positive, but noisy relationship exists between fouls and yellow cards for both home and away teams, suggesting that more aggressive teams do tend to be penalized, but not uniformly.

* The wide scatter indicates referee discretion plays a substantial role; identical foul counts can produce very different card totals depending on match context or officiating style.

* Home and away trends are similar, implying that card issuance is not dramatically skewed toward one side based purely on fouls committed.

* Rare but notable instances of many cards with relatively few fouls suggest isolated high-intensity incidents rather than consistent physical play.


## 3.4 Betting Market Behavior

### 3.4.1 Distribution of Betting Market Odds

Understanding the distribution of betting odds is essential for interpreting how bookmakers assess match competitiveness. Lower odds indicate strong favorites, while higher odds suggest underdogs. Because multiple bookmakers are included (Bet365, Bet&Win, Interwetten, William Hill, VC Bet), we analyze the typical pricing ranges for home win, draw, and away win outcomes across the market.

In [ ]:
odds_cols_H = ["B365H"]
odds_cols_D = ["B365D"]
odds_cols_A = ["B365A"]

plt.figure(figsize=(16,5))

# Home odds
plt.subplot(1,3,1)
sns.histplot(soccer_df[odds_cols_H].melt()["value"], bins=30, kde=True, color="blue")
plt.title("Distribution of Home Win Odds")
plt.xlabel("Home Odds")

# Draw odds
plt.subplot(1,3,2)
sns.histplot(soccer_df[odds_cols_D].melt()["value"], bins=30, kde=True, color="gray")
plt.title("Distribution of Draw Odds")
plt.xlabel("Draw Odds")

# Away odds
plt.subplot(1,3,3)
sns.histplot(soccer_df[odds_cols_A].melt()["value"], bins=30, kde=True, color="red")
plt.title("Distribution of Away Win Odds")
plt.xlabel("Away Odds")

plt.tight_layout()
plt.show()

Takeaways:

* Home odds cluster more tightly around lower values compared to away odds, reflecting the consistent home-field advantage observed earlier in the dataset.

* Draw odds are distributed in a narrower band, indicating that bookmakers see draws as less sensitive to team strength differences and more influenced by league-wide scoring patterns.

* Away odds show a long right tail, capturing matches where away teams are substantial underdogs; this asymmetry highlights that strong away favorites are relatively rare.

* The separation between home, draw, and away distributions suggests that the market builds in structural expectations about match dynamics that can be leveraged for predictive modeling.

* The consistency across all bookmakers suggests general market efficiency, most betting firms price matches similarly, with differences arising mainly from margin structures rather than qualitative disagreement.


### 3.4.2 Betting Odds vs Actual Match Outcomes

In this subsection, we test how well pre-match odds align with actual match outcomes. By comparing the odds of home, draw, and away results with the recorded match result (FTR), we can identify whether favorites tend to win as expected, how often underdogs upset the market, and whether certain odds ranges consistently overperform or underperform expectations.

In [ ]:
plt.figure(figsize=(16,5))

# Home odds vs outcome
plt.subplot(1,3,1)
sns.boxplot(x=soccer_df["FTR"], y=soccer_df["B365H"], palette="Set2")
plt.title("Home Odds vs Match Result (Bet365)")
plt.xlabel("Final Result (FTR)")
plt.ylabel("Home Odds (B365H)")

# Draw odds vs outcome
plt.subplot(1,3,2)
sns.boxplot(x=soccer_df["FTR"], y=soccer_df["B365D"], palette="Set2")
plt.title("Draw Odds vs Match Result (Bet365)")
plt.xlabel("Final Result (FTR)")
plt.ylabel("Draw Odds (B365D)")

# Away odds vs outcome
plt.subplot(1,3,3)
sns.boxplot(x=soccer_df["FTR"], y=soccer_df["B365A"], palette="Set2")
plt.title("Away Odds vs Match Result (Bet365)")
plt.xlabel("Final Result (FTR)")
plt.ylabel("Away Odds (B365A)")

plt.tight_layout()
plt.show()


Takeaways:

* Matches that end in home wins tend to have lower home odds, reflecting that on average, bookmakers correctly identify most strong home favorites.

* Away wins are associated with higher away odds, demonstrating that many away victories occur in matches where the away side was not expected to win.

* Draw outcomes occur across a relatively boring range of odds, confirming the difficulty of pricing draws and the inherent unpredictability of low-scoring matches.

* Outliers with extremely high winning odds represent classic “upset” matches, where the model and market differ substantially. These are difficult to predict and most likely won't be accounted for by models.

* These plots validate the predictive power of betting odds on average and justify their use as baseline features or benchmarks in outcome prediction models.


### 3.4.3 Implied Probability Analysis

Betting odds can be converted into implied probabilities, which express the market’s estimated chance of each outcome occurring. Because bookmakers include profit margins (“overround”), the raw probabilities will sum to more than 1. Normalizing them allows us to compare predicted probability distributions to actual match outcomes, offering insight into market calibration and potential inefficiencies.

In [ ]:
# Compute implied probabilities for Bet365
soccer_df["pH"] = 1 / soccer_df["B365H"]
soccer_df["pD"] = 1 / soccer_df["B365D"]
soccer_df["pA"] = 1 / soccer_df["B365A"]

# Normalize probabilities (remove bookmaker margin)
total = soccer_df["pH"] + soccer_df["pD"] + soccer_df["pA"]
soccer_df["pH_norm"] = soccer_df["pH"] / total
soccer_df["pD_norm"] = soccer_df["pD"] / total
soccer_df["pA_norm"] = soccer_df["pA"] / total

plt.figure(figsize=(14,5))
sns.kdeplot(soccer_df["pH_norm"], shade=True, label="Home")
sns.kdeplot(soccer_df["pD_norm"], shade=True, label="Draw")
sns.kdeplot(soccer_df["pA_norm"], shade=True, label="Away")
plt.title("Normalized Implied Probability Distributions (Bet365)")
plt.xlabel("Normalized Probability")
plt.legend()
plt.show()

soccer_df.head()

Takeaways:

* Normalized home win probabilities dominate the right side of the distribution, indicating that bookmakers consistently assign higher chances to home teams.

* Draw probabilities form a very compact central distribution, manifesting as a spike, reinforcing the idea that the draw is priced around league-wide scoring tendencies rather than team-level dynamics.

* Away win probabilities are skewed toward low values, reflecting the rarity of away dominance and aligning with earlier performance patterns.

* The separation of the three curves demonstrates that bookmakers incorporate structural features of football, home-field advantage, low-scoring draw likelihood, and away underperformance, directly into their pricing models.

* These normalized probabilities can serve as high-quality model features or baselines when evaluating the accuracy of predictive models for match outcomes.


##3.5 Trends across seasons


In [ ]:
# Get seasons in chronological order
seasons = sorted(soccer_df['Season'].unique())

seasons

### 3.5.1 Home Win Odds Trends Across Seasons

Examining the evolution of average betting odds across all three outcomes reveals important trends in match competitiveness over time. The comparison between home win, draw, and away win odds shows how home advantage, competitive balance, and match predictability have changed across seasons, providing insights into bookmakers' evolving assessment of league dynamics and helping identify periods of greater or lesser competitive parity.

In [ ]:
# Combined Odds Trends Across Seasons
plt.figure(figsize=(14, 8))

season_home_odds = soccer_df.groupby('Season')['B365H'].mean()
season_draw_odds = soccer_df.groupby('Season')['B365D'].mean()
season_away_odds = soccer_df.groupby('Season')['B365A'].mean()

plt.plot(seasons, season_home_odds, marker='o', linewidth=2.5,
         markersize=8, color='#3498db', label='Home Win')
plt.plot(seasons, season_draw_odds, marker='s', linewidth=2.5,
         markersize=8, color='#95a5a6', label='Draw')
plt.plot(seasons, season_away_odds, marker='^', linewidth=2.5,
         markersize=8, color='#e74c3c', label='Away Win')

plt.title('Average Betting Odds Trends by Season', fontsize=15, fontweight='bold')
plt.xlabel('Season', fontsize=12)
plt.ylabel('Average Odds', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Takeaways:

* All three odds types (home, draw, away) show a gradual upward trend across seasons, suggesting that matches have become less predictable over time. Higher odds reflect increased uncertainty in outcomes.

* Home win odds rise slightly but steadily, indicating that bookmakers perceive a weakening of traditional home-field advantage. The growing competitiveness of away sides may contribute to this shift.

* Away win odds fluctuate more sharply than home odds, reflecting larger variations in away-team strength and the broader range of potential underdog scenarios.

* Overall, the upward drift in odds across all outcomes points to a trend toward greater parity and reduced dominance gaps in the league, which aligns with the decrease in home win rates seen later in Section 3.4.

###3.5.2 Total Matches and Home Win Rate Trend

This chart tracks the home win rate trend across all seasons in the dataset, showing how the percentage of home team victories has evolved over time. The trend reveals whether home advantage has remained consistent or experienced significant fluctuations, providing insights into the stability of home field advantage and potential shifts in competitive dynamics across different seasons.

In [ ]:
plt.figure(figsize=(24,12))
plt.subplot(2, 3, 6)

home_win_rate = soccer_df.groupby('Season')['FTR'].apply(
    lambda x: (x == 'H').mean() * 100
)
seasons_with_data = home_win_rate.index

color = '#e74c3c'
plt.plot(seasons_with_data, home_win_rate, marker='o', color=color, linewidth=2.5,
         markersize=8, label='Home Win %')
plt.ylabel('Home Win Rate (%)', fontsize=11)
plt.xlabel('Season', fontsize=11)
plt.title('Home Win Rate by Season', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


Takeaways:

* Home win rates show a noticeable decline over the dataset, falling from roughly 50% in the early seasons to around 44% in more recent years. This indicates a meaningful erosion of home-field advantage.

* The downward pattern suggests that away teams have become more competitive, potentially due to improved travel logistics, tactical adjustments, or a narrowing skill gap.

* Season-to-season volatility signals that home advantage is not constant and may be influenced by external factors such as scheduling, rule changes, or evolving play styles.

* Despite fluctuations, the long-term trend clearly points to a structural shift toward more balanced competition, which aligns with the gradual increase in betting odds seen in Section 3.4.1.

* This weakening home advantage is important context for modeling: outcome probabilities today may differ systematically from those in earlier seasons, highlighting the value of including season-based features or acknowledging temporal changes in predictive patterns.

# Part 4: Feature Engineering & Preprocessing

Before building any predictive models, we performed additional preprocessing and feature-engineering steps to ensure that our dataset was structured, consistent, and ready for classification tasks. These steps refine the cleaned data from Part 2 and allow us to extract meaningful predictive information from the match records.

Note that earlier we added a travel distance feature to represent a location based feature.

## 4.1 Feature Engineering

**Creating additional match-level features**

We engineered new variables that capture important aspects of match dynamics and bookmaker expectations.



In [ ]:
# Total goals scored in the match
soccer_df["TotalGoals"] = soccer_df["FTHG"] + soccer_df["FTAG"]

# Goal difference between home and away teams
soccer_df["GoalDiff"] = soccer_df["FTHG"] - soccer_df["FTAG"]

# Indicator for whether bookmakers favored the home team
soccer_df["HomeFavored"] = (soccer_df["B365H"] < soccer_df["B365A"]).astype(int)

# Binary flag for whether a match is considered high scoring (>= 3 total goals)
soccer_df["HighScoring"] = (soccer_df["TotalGoals"] >= 3).astype(int)

## 4.2 Encoding Categorical Variables



**One-hot encoding season and league identifiers**

To prepare our categorical fields for modeling, we converted them into binary indicator variables.

In [ ]:
# One-hot encoding categorical columns
soccer_df = pd.get_dummies(
    soccer_df,
    columns=["League", "Season"],
    drop_first=True
)

## 4.3 Correlation

**Inspecting correlations to avoid redundant features**

We computed correlations to ensure no pair of variables was excessively collinear, especially among odds features.


In [ ]:
# Correlation matrix for key numerical features
soccer_df.corr(numeric_only=True)

## 4.4 Addressing Class Imbalance

**Checking distribution of match outcomes**

Match outcomes are not evenly distributed, which will guide our modeling strategy.

In [ ]:
# Distribution of target classes (H, D, A)
soccer_df["FTR"].value_counts(normalize=True)

## 4.5 Scaling Numerical Features

**Standardizing numerical predictors for modeling**

We scaled numerical variables so models like logistic regression are not influenced by differences in measurement scale.

In [ ]:
# Selecting numeric columns
numeric_cols = soccer_df.select_dtypes(include=["float64", "int64"]).columns

# Initializing and applying the scaler
scaler = StandardScaler()
soccer_df[numeric_cols] = scaler.fit_transform(soccer_df[numeric_cols])

These preprocessing steps transform our refined dataset into a structure suitable for predictive modeling. By engineering new features, encoding categorical variables, checking correlations, addressing outcome imbalance, and scaling numerical predictors, we ensure that our data is clean, informative, and compatible with downstream machine learning algorithms.

# Part 5: Baseline Model (Logistic regression using only odds features)

At this point, after completing data loading and preprocessing, we are ready to establish a baseline predictive model for match outcomes.
The baseline is essential because it allows us to compare all future models against a simple, interpretable benchmark, ensuring that any additional model complexity is justified.

We chose logistic regression using only bookmaker odds as features. Betting odds encode expert market expectations and have been shown to contain substantial predictive signal, making them a natural starting point for our classification task.

Before fitting this model, we address the inherent **class imbalance** in our dataset. Home wins are much more frequent than draws or away wins, so we apply class_weight="balanced" to ensure that minority outcomes receive proportional importance during training.


In [ ]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    multi_class="multinomial"
)

rf_classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

## 5.1 Preparing Baseline Features (Odds Only)

We restrict our feature set to the three primary 1X2 odds:

* B365H: Bet365 Home Win odds

* B365D: Bet365 Draw odds

* B365A: Bet365 Away Win odds

These odds are available across most seasons and represent the simplest, non-engineered feature group.

In [ ]:
# Extract odds-only baseline features
baseline_features = soccer_df[['B365H', 'B365D', 'B365A']]

# Extract the target variable (FTR = match outcome)
baseline_target = soccer_df['FTR']


## 5.2 Train–Validation Split (Time-Based)

To avoid data leakage, we split the dataset by chronological order instead of shuffling.

In [ ]:
# Time-based train/validation split
train_size = int(0.8 * len(soccer_df))

X_train = baseline_features.iloc[:train_size]
X_val   = baseline_features.iloc[train_size:]

y_train = baseline_target.iloc[:train_size]
y_val   = baseline_target.iloc[train_size:]

## 5.3 Baseline Logistic Regression Model

We use a multinomial logistic regression model because there are three possible classes:
Home Win (H), Draw (D), Away Win (A).

In [ ]:
# Fit logistic regression on odds-only features
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

# Predict on validation set
y_pred = log_reg.predict(X_val)

## 5.4 Baseline Evaluation

We compute Accuracy for the Logistic Regression Mode as done below:

In [ ]:
# Evaluate baseline model
accuracy = accuracy_score(y_val, y_pred)

print("Baseline Logistic Regression (Odds Only)")
print("Validation Accuracy:", accuracy)

soccer_df.to_csv("output_analysis.csv", index=False)





Because our dataset is imbalanced across match outcomes (Home wins > Draws > Away wins), accuracy alone is insufficient for evaluating model performance. We compute additional metrics including precision, recall, macro F1-score, and the confusion matrix. These metrics provide deeper insight into how the model performs across each class, especially the minority classes (Draw/Away).

In [ ]:
# Macro F1-score (treats all classes equally, important for imbalanced data)
macro_f1 = f1_score(y_val, y_pred, average='macro')

# Print full classification report (precision, recall, f1 per class)
print("\nClassification Report (Odds-Only Baseline):")
print(classification_report(y_val, y_pred))

print(f"Macro F1-Score: {macro_f1:.4f}")

# Confusion matrix visualization
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, cmap='Blues',
            xticklabels=['H', 'D', 'A'],
            yticklabels=['H', 'D', 'A'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – Baseline Logistic Regression")
plt.show()

## 5.5 Interpretation of Baseline Performance

Based on our evaluation using **Accuracy**, **Macro F1-Score**, the **classification report**, and the **confusion matrix**, we observe the following:


### **Performance Summary**
- **Accuracy:** 47.32%
- **Macro F1-Score:** **0.3164**

Accuracy alone suggests that the odds-only logistic regression baseline performs reasonably well.  
However, the Macro F1-Score and class-specific metrics reveal important weaknesses that accuracy hides.


### **Key Takeaways**

- **Bookmaker odds encode meaningful predictive power.**  
  Even a simple linear model reaches ~47% accuracy, indicating that market odds contain strong signals about match outcomes.

- **The model heavily favors the Home Win (H) class.**  
  From the classification report:
  - **Home Wins:** High recall (~0.91), moderate precision  
  - **Away Wins:** Low recall (~0.25)  
  - **Draws:** Precision = 0.00, Recall = 0.00 (model almost never predicts Draw)

  This bias reflects:
  1. **Real-world home-field advantage**  
  2. **Class imbalance in the dataset (H > D > A)**  
  3. **Bias encoded in the odds themselves**

- **Draws remain the hardest class to predict**, consistent with soccer analytics literature.  
  Draw outcomes are inherently noisy and harder to model.

- **Macro F1-Score exposes imbalance issues hidden by accuracy.**  
  Because Macro-F1 treats each class equally, the low score (0.3164) highlights that the model **performs poorly on minority classes**, especially Draws.

### **Confusion Matrix**

The confusion matrix further highlights the weaknesses of the baseline model. The logistic regression classifier **never predicts Draw (D)** for any sample, as shown by the zero-valued column for the D prediction. Instead, the model overwhelmingly defaults to predicting **Away Win (A)**, even when the true label is Home or Draw. This collapse toward a single dominant class explains the extremely low Macro F1-Score (0.3164) and the poor recall and precision for the minority classes. While accuracy appears moderate, the confusion matrix clearly shows that the baseline model fails to meaningfully distinguish between the three outcomes, especially Draws, which it predicts as A in nearly all cases. This reinforces the need for more flexible models capable of handling nonlinear boundaries and class imbalance.


### **Limitations of the Baseline Model**

Although logistic regression provides a simple and interpretable benchmark, it suffers from several limitations:

- **Assumes linear relationships** between odds and outcomes, which may be too restrictive for the nonlinear dynamics of football matches.
- Uses **only bookmaker odds**, ignoring richer match-context and performance features (shots, fouls, goal difference, etc.).
- **Highly sensitive to class imbalance**, leading to poor recall on Draw and Away classes despite class-weight adjustments.
- Cannot capture **interactions or complex feature relationships**, unlike Random Forests or Neural Networks.

These limitations motivate moving to more flexible models that can better handle nonlinear interactions, imbalance, and richer feature sets.


# Part 6: Advanced Models with Full Feature Set


In this section, we expand beyond the baseline by leveraging the complete set of engineered features from Part 4. We will train and evaluate three advanced machine learning models: Decision Tree, Random Forest, and Neural Network. Each model will be assessed using Accuracy to enable fair comparison.

##6.1 Prepare Full Feature Set

We prepare the complete feature set by selecting all engineered and scaled features from Part 4, excluding only the target variable (FTR) and non-predictive columns like team names and dates.

In [ ]:
# Define columns to drop (target, leakage, or non-predictive)
target_cols = ['FTR', 'HTR']  # target variables
result_cols = ['FTHG', 'FTAG', 'HTHG', 'HTAG', 'GoalDiff',
               'TotalGoals', '1stHalfGoals', '2ndHalfGoals']  # leakage columns
non_predictive_cols = ['Date', 'HomeTeam', 'AwayTeam', 'Referee']  # identifiers + metadata

columns_to_drop = target_cols + result_cols + non_predictive_cols

# Remove the unwanted columns
full_features = soccer_df.drop(columns=columns_to_drop, errors='ignore')

# Drop any object (string) columns that remain — ML models require numerical input
object_cols = full_features.select_dtypes(include=['object']).columns
full_features = full_features.drop(columns=object_cols)

# Create target vector
full_target = soccer_df['FTR'].copy()

# Handle missing numeric values by imputing with column mean
if full_features.isnull().sum().sum() > 0:
    full_features = full_features.fillna(full_features.mean())

# Replace infinite values with NaN, fill, clean
if np.isinf(full_features.select_dtypes(include=[np.number])).sum().sum() > 0:
    full_features = full_features.replace([np.inf, -np.inf], np.nan)
    full_features = full_features.fillna(full_features.mean())

# Time-based split (avoids leakage from future data)
train_size = int(0.8 * len(soccer_df))

X_train_full = full_features.iloc[:train_size]
X_val_full   = full_features.iloc[train_size:]

y_train_full = full_target.iloc[:train_size]
y_val_full   = full_target.iloc[train_size:]

In [ ]:
# Encode target variable to numeric (0, 1, 2)
le = LabelEncoder()
y_train_full_encoded = le.fit_transform(y_train_full)
y_val_full_encoded = le.transform(y_val_full)

print(f" Target encoded: {le.classes_} -> {list(range(len(le.classes_)))}")
print(f" y_train_full dtype: {y_train_full_encoded.dtype}")

# Replace original target variables with encoded versions
y_train_full = y_train_full_encoded
y_val_full = y_val_full_encoded

##6.2 Random Forest Classifier

While our baseline logistic regression model using betting odds provides a strong benchmark, reflecting the predictive power of bookmaker expectations, it is inherently linear and limited in the interactions it can model. To move beyond these constraints, we introduced a Random Forest classifier, a non-linear ensemble method capable of capturing complex relationships across team-level statistics, shot metrics, disciplinary patterns, and normalized betting features. Unlike logistic regression, Random Forests do not assume linear separability and can naturally model features that interact in non-additive ways (e.g., how shots on target interact with home-field dynamics or how fouls combine with match tempo). Random Forests also provide built-in feature importance scores, giving us interpretable insight into which engineered variables meaningfully contribute to predicting match outcomes—something neural networks and other black-box models cannot offer as transparently. Because of their robustness, ability to handle mixed-scale features, and resistance to overfitting through averaging across many trees, Random Forests serve as a strong next step in improving accuracy over the odds-only baseline.

In [ ]:
# Initialize a Random Forest classifier
# n_estimators: number of trees
# max_depth: limits tree depth to reduce overfitting
# min_samples_split / min_samples_leaf: enforce larger leaf nodes for smoother decision boundaries
# random_state: ensures reproducible results
# n_jobs=-1: uses all CPU cores for faster training

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=50,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

# Train the model on the full feature training set
rf_classifier.fit(X_train_full, y_train_full)

# Generate predictions on the validation split
y_pred_rf = rf_classifier.predict(X_val_full)

# Compute validation accuracy (primary metric for quick comparison)
rf_accuracy = accuracy_score(y_val_full, y_pred_rf)

# Report the result
print("Random Forest Classifier Results")
print(f"Validation Accuracy: {rf_accuracy:.4f}")


In addition to accuracy, we evaluate the Random Forest using macro F1 and a confusion matrix to account for class imbalance. Macro F1 weights each class equally, so poor performance on the minority “Draw” or “Away Win” classes cannot be hidden by the frequent “Home Win” class. The confusion matrix lets us see where the model makes systematic errors (e.g., confusing draws with home wins). This richer set of metrics prevents us from relying solely on accuracy and aligns with our earlier discussion of class imbalance.

In [ ]:
# Additional evaluation metrics for Random Forest
rf_cm = confusion_matrix(y_val_full, y_pred_rf, labels=[0, 1, 2])
rf_macro_f1 = f1_score(y_val_full, y_pred_rf, average='macro')

print("Random Forest – additional metrics")
print("Macro F1:", rf_macro_f1)
print("Confusion matrix:\n", rf_cm)
print("\nClassification report:\n", classification_report(y_val_full, y_pred_rf))

###6.2.1 Random Forest Evaluation: Feature Importance & Hyperparameter Tuning

Random Forest feature importance provides a more stable measure of which inputs matter most because it averages importance scores across all trees, reducing noise from any single model. Each tree evaluates how much a feature improves the purity of its splits, and features that consistently help separate match outcomes receive higher importance scores. By aggregating these effects, the Random Forest highlights which engineered variables, such as shots on target or normalized betting probabilities, truly drive predictive performance. This gives us a clearer understanding of the underlying drivers of match results and helps validate whether our feature engineering aligns with meaningful football dynamics.

In [ ]:
# Extract averaged feature importance from all trees
feature_importance_rf = pd.DataFrame({
    'feature': X_train_full.columns,
    'importance': rf_classifier.feature_importances_
}).sort_values('importance', ascending=False)

# Visualize top 15 features
plt.figure(figsize=(10, 6))
plt.barh(
    feature_importance_rf.head(15)['feature'],
    feature_importance_rf.head(15)['importance'],
    color='forestgreen',
    edgecolor='darkgreen',
    linewidth=1.2
)
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Top 15 Most Important Features - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Print out the top 10 most important features, and their importance
print("\nTop 10 Most Important Features:")
print(feature_importance_rf.head(10).to_string(index=False))

The initial Random Forest used hand-chosen parameters and already improved over the odds-only baseline, but training/validation accuracy suggested room for improvement. Because Random Forests are sensitive to the number of trees and maximum depth, we perform a focused GridSearchCV over n_estimators and max_depth. This gives us a systematic, cross-validated way to trade off model capacity and overfitting instead of guessing hyperparameters.

In [ ]:
# Perform a limited hyperparameter search for Random Forest

# n_estimators: number of trees in the forest
# max_depth: maximum depth allowed for each tree
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [15, 20]
}

# Set up GridSearchCV to run cross-validated model tuning
# cv=3: 3-fold cross-validation
# scoring='accuracy': optimize for classification accuracy
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Run the grid search on the training data
grid_search.fit(X_train_full, y_train_full)

# Display the best hyperparameters and their CV validation performance
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# Extract the best-performing Random Forest model
rf_classifier = grid_search.best_estimator_


The tuned Random Forest slightly increases validation accuracy compared to the untuned version and, more importantly, yields a more balanced macro F1 across classes. This indicates that adding trees while capping depth helps the model capture additional nonlinear structure without severely overfitting. We use this tuned configuration in all subsequent comparisons.

## 6.3 Interpretation of Random Forest Model

Our Random Forest model provides a substantial improvement over the odds-only logistic regression baseline, demonstrating the value of adding engineered match-performance features (shots, cards, normalized odds, etc.). Below are the major insights from the full evaluation:

#### **Overall Performance**
- **Validation Accuracy:** ~0.543  
- **Macro F1:** ~0.507  
- Both metrics outperform the baseline model’s **47.3% accuracy** and **0.316 macro F1**, reflecting significantly better class balance and predictive power.

The improvement in **macro F1** is especially important: it shows that Random Forest performs better across *all* classes rather than relying heavily on the majority “Home Win” class.

#### **Class-Wise Behavior**
From the classification report:

- **Home Win (H)**  
  - Precision: 0.58  
  - Recall: 0.38  
  - F1: 0.46  
  Although still imperfect, performance on Home wins is more balanced than the baseline’s extreme overprediction of this class.

- **Draw (D)**  
  - Precision: 0.41  
  - Recall: 0.39  
  - F1: 0.40  
  Draw prediction—historically the hardest outcome—improves meaningfully over the baseline (which had **0.00** F1 for Draws).

- **Away Win (A)**  
  - Precision: 0.59  
  - Recall: 0.74  
  - F1: 0.66  
  The model is especially strong at identifying Away wins, indicating that the engineered performance features add useful discriminatory power.

#### **Confusion Matrix Insights**
The confusion matrix reveals clearer, more structured prediction behavior compared to the baseline:

- The model **no longer collapses all predictions into a single class** (as logistic regression did).  
- Draws and Away wins receive substantial true positives.  
- Most remaining errors reflect intuitive confusions:
  - H ↔ A misclassifications occur when match features are similar between balanced teams.
  - Draws are still sometimes misclassified as H or A, consistent with their inherent noise and low frequency.

This demonstrates that Random Forest captures **real football dynamics** rather than relying solely on betting market signals.

#### **Feature Importance – What the Model Learned**
Top features such as **HighScoring**, **HST**, **AST**, and normalized odds (**pH_norm**, **pA_norm**) emerge as the strongest predictors.

These results validate:
- The importance of engineered match-stat features (e.g., shots on target).  
- The model’s ability to combine betting signals with on-field performance.  
- That our features meaningfully contribute to distinguishing match outcomes.

#### **Impact of Hyperparameter Tuning**
Tuning significantly strengthened the model:

- **Best parameters:**  
  - `n_estimators = 150`  
  - `max_depth = 15`  
- **Cross-validated score:** 0.5603

Key improvements from tuning:
- Deeper but controlled trees improved class separation.  
- More trees reduced variance and stabilized performance.  
- Macro F1 became more balanced across classes, reflecting improved minority-class prediction.

### **Summary**
The Random Forest model marks a major step forward from the baseline:

- **Higher accuracy**  
- **Massively improved macro F1**  
- **Better performance on minority classes (Draw, Away)**  
- **Meaningful, interpretable feature importance**  
- **Improved class balance and fewer systematic misclassifications**  

Overall, Random Forest provides a **strong, robust, and interpretable** upgrade over logistic regression—capturing nonlinear relationships and producing more realistic classification behavior across match outcomes.

##6.4 Neural Network

After evaluating Logistic Regression and Random Forest, we turned to a Neural Network to explore whether more complex, nonlinear representations could further improve predictive performance. While Random Forests capture moderately nonlinear relationships, they are still limited by tree-based partitioning. Neural Networks, by contrast, can model high-dimensional interactions between features, potentially uncovering deeper patterns related to match dynamics, team tendencies, and contextual factors. A neural architecture also allows continuous optimization rather than discrete splits, which may better fit our engineered numeric features. However, Neural Networks require careful preprocessing and tuning, so the following sections outline how we prepared the data, trained the initial model, refined it through grid search, and validated its learning behavior.

Defining the Neural Network:

We construct a Multi-Layer Perceptron (MLP) with three hidden layers (128 to 64 to 32). This decreasing depth architecture provides strong representational capacity while encouraging generalization. ReLU activation enables nonlinear learning, and the Adam optimizer accelerates convergence. Early stopping monitors validation performance and halts training when improvement plateaus, preventing overfitting.

In [ ]:
# Additional data type check and conversion for Neural Network
print("Checking and converting data types for Neural Network...")

# Convert all columns to numeric, forcing errors to NaN
X_train_full = X_train_full.apply(pd.to_numeric, errors='coerce')
X_val_full = X_val_full.apply(pd.to_numeric, errors='coerce')

# Fill any NaN values created during conversion
X_train_full = X_train_full.fillna(0)
X_val_full = X_val_full.fillna(0)

# Ensure all values are float64
X_train_full = X_train_full.astype(np.float64)
X_val_full = X_val_full.astype(np.float64)

# Validate data types
print(f"X_train_full dtype: {X_train_full.dtypes.unique()}")
print(f"All numeric: {X_train_full.select_dtypes(include=[np.number]).shape[1] == X_train_full.shape[1]}")


The neural network is trained on the full training set, after which predictions are generated on the validation split. We report validation accuracy as the primary performance metric, alongside the number of training iterations and the best internal validation score achieved before early stopping. These diagnostics help assess stability, convergence, and overfitting risk.

In [ ]:
# Define the Neural Network Classifier
nn_classifier = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),   # Three hidden layers with decreasing size
    activation='relu',                  # ReLU activation for nonlinear learning
    solver='adam',                      # Adam optimizer for efficient training
    alpha=0.001,                        # L2 regularization strength
    learning_rate='adaptive',           # Learning rate decreases when loss plateaus
    max_iter=500,                       # Maximum number of training iterations
    early_stopping=True,                # Stop early if validation doesn't improve
    validation_fraction=0.1,            # Portion of training data used for validation
    random_state=42,
    verbose=False
)

# Train the neural network on the full training dataset
nn_classifier.fit(X_train_full, y_train_full)

# Predict class labels on validation set
y_pred_nn = nn_classifier.predict(X_val_full)

# Compute validation accuracy
nn_accuracy = accuracy_score(y_val_full, y_pred_nn)

# Print model performance results
print("Neural Network (MLP) Results\n")
print(f"Validation Accuracy: {nn_accuracy:.4f}")                 # Final model accuracy
print(f"\nNumber of iterations: {nn_classifier.n_iter_}")       # How many epochs it trained before stopping
print(f"Best validation score: {nn_classifier.best_validation_score_:.4f}")  # Best internal val accuracy


To strengthen model performance, we run a limited grid search exploring different hidden-layer configurations and L2 regularization strengths (alpha). Using 3-fold cross-validation helps identify architecture shapes and regularization settings that generalize beyond a single train/validation split. The best-performing model is then selected for final evaluation.

In [ ]:
# Define a limited hyperparameter grid for the MLPClassifier
param_grid = {
    'hidden_layer_sizes': [
        (128, 64, 32),     # Deep network with 3 layers
        (128, 64),         # Medium network with 2 layers
        (256, 128, 64)     # Wider network for more capacity
    ],
    'alpha': [0.0001, 0.001]  # L2 regularization strengths
}

# Perform Grid Search with 3-fold cross-validation
grid_search = GridSearchCV(
    MLPClassifier(
        activation='relu',
        solver='adam',
        learning_rate='adaptive',
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
        verbose=False
    ),
    param_grid,             # The hyperparameters to try
    cv=3,                   # 3-fold cross-validation
    scoring='accuracy',     # Use accuracy as evaluation metric
    n_jobs=-1,
    verbose=1
)

# Fit grid search on training data
grid_search.fit(X_train_full, y_train_full)

# Retrieve the best-performing model found by the search
nn_classifier = grid_search.best_estimator_

# Print summary results from grid search
print("Grid Search Results\n")
print(f"Best parameters: {grid_search.best_params_}")          # Best hyperparameter combination
print(f"Best CV score: {grid_search.best_score_:.4f}")         # Best cross-validation accuracy


In [ ]:
# Additional evaluation metrics for Neural Network
nn_macro_f1 = f1_score(y_val_full, y_pred_nn, average='macro')
nn_conf_mat = confusion_matrix(y_val_full, y_pred_nn)

print(f"Neural Network (MLP) Results\n")
print(f"Validation Accuracy: {nn_accuracy:.4f}")
print(f"Macro F1 Score:      {nn_macro_f1:.4f}")
print("\nConfusion Matrix (rows = true, cols = predicted):")
print(nn_conf_mat)

### Neural Network Evaluation and Limitations

The tuned neural network achieves the **highest validation performance** among all models, reaching  
**55.67% validation accuracy** and a **macro F1-score of 0.5316**, slightly outperforming the Random Forest and substantially improving upon the odds-only baseline. These gains indicate that the neural network is able to capture richer nonlinear relationships between match statistics and outcomes, particularly when compared to linear and tree-based baselines.

The **confusion matrix** shows that the neural network improves prediction of the minority classes—especially **Draw** and **Away Win**—compared to earlier models. Home wins remain the easiest class overall, but unlike the baseline logistic regression (which collapses almost entirely into predicting Away), the neural network exhibits more balanced behavior across classes.

The **learning curve** decreases smoothly and flattens before early stopping activates, suggesting healthy optimization without overfitting. This indicates that the model converges efficiently while maintaining generalization. Hyperparameter tuning further confirmed that a **simpler 2-layer architecture (128, 64)** outperformed both deeper and wider networks, balancing capacity with regularization (best alpha = 0.0001).

However, the neural network also introduces key limitations.  
Unlike Random Forests, the MLP is **less interpretable**, sensitive to scaling and hyperparameter choices, and its probability calibration may not be as stable. These characteristics make the neural network a strong but more opaque model—best used when predictive performance outweighs interpretability concerns.

### 6.4.1 Neural Network Learning Curve

Finally, we examine the neural network’s learning curve, a plot of training loss across epochs. A steadily decreasing curve indicates healthy optimization, while early flattening or oscillation may signal underfitting or overfitting. Because early stopping is enabled, the curve also confirms whether the model halted at an appropriate time, balancing accuracy and generalization.

In [ ]:
# Part 6.3.1: Neural Network Learning Curve

# Plot training loss across iterations
plt.figure(figsize=(10, 6))
plt.plot(nn_classifier.loss_curve_, linewidth=2, color='darkblue')
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Neural Network Learning Curve', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal training loss: {nn_classifier.loss_curve_[-1]:.4f}")
print(f"Converged after {len(nn_classifier.loss_curve_)} iterations")

In [ ]:
# Model Results Summary Overall

# Baseline Model (Logistic Regression on Odds Only)
baseline_accuracy = accuracy  # From Part 5
baseline_macro_f1 = f1_score(y_val, y_pred, average='macro')
baseline_cm = confusion_matrix(y_val, y_pred)

print("\nBaseline Model (Logistic Regression – Odds Only)")
print(f"  Accuracy:     {baseline_accuracy:.4f}")
print(f"  Macro F1:     {baseline_macro_f1:.4f}")
print(f"  Confusion Matrix:\n{baseline_cm}")


# Random Forest (Using Tuned Model)
y_pred_rf = rf_classifier.predict(X_val_full)
rf_accuracy = accuracy_score(y_val_full, y_pred_rf)
rf_macro_f1 = f1_score(y_val_full, y_pred_rf, average='macro')
rf_cm = confusion_matrix(y_val_full, y_pred_rf)

print("\nRandom Forest (Tuned)")
print(f"  Best params:  n_estimators=150, max_depth=15")
print(f"  CV accuracy:  0.5603")
print(f"  Validation Accuracy: {rf_accuracy:.4f}")
print(f"  Macro F1:     {rf_macro_f1:.4f}")
print(f"  Confusion Matrix:\n{rf_cm}")

# Neural Network (Using Tuned Model)
y_pred_nn = nn_classifier.predict(X_val_full)
nn_accuracy = accuracy_score(y_val_full, y_pred_nn)
nn_macro_f1 = f1_score(y_val_full, y_pred_nn, average='macro')
nn_cm = confusion_matrix(y_val_full, y_pred_nn)

print("\nNeural Network (Tuned)")
print(f"  Best params:  hidden_layers=(128, 64), alpha=0.0001")
print(f"  CV accuracy:  0.5748")
print(f"  Validation Accuracy: {nn_accuracy:.4f}")
print(f"  Macro F1:     {nn_macro_f1:.4f}")
print(f"  Confusion Matrix:\n{nn_cm}")


## 6.5 Model Results Summary

### **Baseline Model — Logistic Regression (Odds Only)**
- **Accuracy:** 0.4745  
- **Macro F1:** 0.3164  
- **Confusion Matrix (rows = true, cols = predicted):**

| True\Pred |   H   |   D   |   A   |
|-----------|-------|-------|-------|
| **H**     | 710   |   0   | 2190  |
| **D**     | 380   |   0   | 2190  |
| **A**     | 400   |   0   | 4000  |

**Interpretation:**  
The model predicts **almost exclusively “Away Win”**, severely underpredicting draws and misclassifying many home wins. Very low macro F1 reflects poor balance across classes despite moderate accuracy driven by class imbalance.

### **Random Forest (Tuned: n_estimators=150, max_depth=15)**
- **Validation Accuracy:** 0.5453  
- **Macro F1:** 0.5058  
- **Confusion Matrix:**

| True\Pred |    H    |    D    |    A    |
|-----------|---------|---------|---------|
| **H**     | 1084    | 759     | 1040    |
| **D**     | 355     | 1032    | 1237    |
| **A**     | 421     | 701     | 3253    |

**Interpretation:**  
The Random Forest produces **far more balanced performance** than the baseline. It recovers a meaningful number of draws and improves prediction of minority classes. Macro F1 roughly **doubles** from the logistic model.

### **Neural Network (Tuned: hidden_layers=(128, 64), alpha=0.0001)**
- **Validation Accuracy:** 0.5624  
- **Macro F1:** 0.5316  
- **Confusion Matrix:**

| True\Pred |    H    |    D    |    A    |
|-----------|---------|---------|---------|
| **H**     | 1470    | 658     | 755     |
| **D**     | 572     | 1045    | 1007    |
| **A**     | 628     | 761     | 2986    |

**Interpretation:**  
The Neural Network provides the **best overall performance**, improving accuracy, draw recall, and away-win classification. Confusion matrix shows the NN recovers more ambiguous outcomes and produces the highest macro F1 among all models.


## 6.6 Key Takeaways

### Overall Model Performance
- The Neural Network is the best-performing model, achieving:
  - 57.5% CV accuracy
  - 56.2% validation accuracy
  - Highest macro F1 (0.53  
- The Random Forest also performs strongly with:
  - 56.0% CV accuracy
  - 54.5% validation accuracy
  - Macro F1: 0.51
- Both models significantly outperform the logistic baseline (47.4% accuracy, macro F1 = 0.32).
- Incorporating match-performance features (shots, fouls, possession stats) leads to ~8–10% improvement over odds-only models.

## Hyperparameter Tuning Results

### Random Forest
- Best configuration:
  - `n_estimators=150`, `max_depth=15`
- Grid Search: 4 combinations × 3 folds = **12 fits**
- Performance notes:
  - Increasing tree count from 100 → 150 yielded consistent gains.
  - max_depth=15 prevented overfitting while still capturing nonlinear structure.
  - Balanced confusion matrix significantly improves performance on minority classes.

### Neural Network
- Best configuration:
  - `hidden_layers=(128, 64)`, `alpha=0.0001`
- Grid Search: 6 combinations × 3 folds = **18 fits**
- Performance notes:
  - 2-layer architecture outperformed all 3-layer networks, suggesting deeper networks overfit.
  - Lower regularization (alpha=0.0001) improved flexibility and accuracy.
  - Learning curve shows smooth, monotonic convergence with early stopping, indicating good generalization.

## Feature Importance Insights (Random Forest)
- Shot-related features dominate predictive power:
  - HighScoring
  - HST (Home Shots on Target)
  - AST (Away Shots on Target)
- Normalized betting odds (pH_norm, pA_norm, pD_norm) remain among the strongest predictors.
- Feature importance confirms that match statistics meaningfully complement market odds, enriching predictive signal.
- Lower-importance features still contribute stability across trees, supporting generalization.

## Model Behavior and Characteristics

### Random Forest
- Offers strong interpretability via feature importance.
- Handles mixed numeric feature scales and non-additive interactions effectively.
- Confusion matrix shows:
  - Major improvement in draw recall vs logistic regression.
  - More balanced classification across all outcomes.

### Neural Network
- Achieves highest accuracy and macro F1.
- Better captures high-dimensional interactions, especially involving shots and normalized probabilities.
- Confusion matrix indicates increased ability to identify borderline matches (draws & away wins).
- Less interpretable (“black-box”), but provides meaningful incremental gains over RF.
- Sensitive to feature scaling and architecture — but stable after tuning.

### Baseline Logistic Regression
- Performs adequately due to strong bookmaker odds signal.
- Severely biased toward predicting Away wins, ignoring minority classes.
- Lowest macro F1 (0.3164), reflecting poor class balance.
- Demonstrates that odds alone cannot fully capture match dynamics.

## Data Quality & Preprocessing Highlights
- Removed goal-dependent features (e.g., FTHG, FTAG, GoalDiff), eliminating data leakage.
- All 46 engineered features are numeric, coherent, and properly scaled.
- Missing values filled, infinite values filtered, ensuring a clean modeling dataset.
- Time-based split prevents future information leaking into training set.

## Key Findings Across All Models
- Shots on Target is the single strongest predictor across models.
- Betting markets are informative but incomplete, stats improve predictive accuracy substantially.
- Simpler NN architectures generalize better than deeper ones for this dataset.
- Regularization tuning matters — small changes in alpha shift accuracy by 1–2%.
- Nonlinear models outperform linear ones by a wide margin when engineered features are available.
- 10% absolute accuracy improvement over baseline demonstrates value of feature engineering + model complexity.

## Computational Considerations
- Random Forest tuning: ~2–3 minutes  
- Neural Network tuning: ~10–15 minutes  
- Both tuning processes produce stable, reusable hyperparameters ideal for production use.
- Given accuracy gains, computational cost is justified.

## **Conclusion**
The tuned Random Forest and Neural Network models provide substantial improvements over the baseline and demonstrate the predictive value of combining betting market information with engineered match statistics. The Random Forest offers interpretability and robustness, while the Neural Network delivers the highest accuracy and macro F1. Together, these results form the empirical foundation for comparing final model performance in Part 7.


# Part 7: Model Comparison and Analysis

After training all three models—Baseline Logistic Regression, Random Forest, and Neural Network—we now evaluate them side-by-side to determine which approach best predicts match outcomes. Each model offers different strengths: logistic regression provides interpretability and serves as a simple benchmark, Random Forest introduces non-linear decision-making and feature importance analysis, and Neural Networks capture deeper and more complex representational patterns.  

In this section, we compare their tuned performance across **accuracy**, **macro F1-score**, and **confusion matrices**, enabling a holistic evaluation of predictive quality. This allows us to identify the most effective model and explain *why* it achieves superior performance.

## 7.1 Performance Comparison

To fairly evaluate model performance, we aggregate the validation accuracies and macro F1-scores of all tuned models into a unified comparison table. This allows us to directly observe the performance gains obtained by moving from a simple baseline (logistic regression using odds only) to more flexible, feature-rich models such as Random Forest and Neural Networks.

Because soccer outcome prediction suffers from **class imbalance** (Home > Away > Draw), accuracy alone is insufficient. We therefore include **macro F1-scores**, which weight all classes equally and reveal how well each model handles minority classes.  

We also compute each model’s **absolute and relative improvement** over the baseline to determine whether increased model complexity translates into meaningful predictive gains. Confusion matrices further highlight which match outcomes each model predicts well and where they still struggle, providing interpretability alongside statistical comparison.

In [ ]:
# Create comparison dataframe with tuned results
model_comparison = pd.DataFrame({
    'Model': [
        'Baseline (Logistic Regression)',
        'Random Forest (Tuned)',
        'Neural Network (Tuned)'
    ],
    'Accuracy': [baseline_accuracy, rf_accuracy, nn_accuracy],
    'Macro_F1': [baseline_macro_f1, rf_macro_f1, nn_macro_f1]
})

print("MODEL PERFORMANCE COMPARISON (WITH HYPERPARAMETER TUNING)\n")
print(model_comparison.to_string(index=False))
print("\n")

# Identify best performing model by Accuracy
best_idx = model_comparison['Accuracy'].idxmax()
best_model = model_comparison.loc[best_idx, 'Model']
best_acc = model_comparison.loc[best_idx, 'Accuracy']
best_f1 = model_comparison.loc[best_idx, 'Macro_F1']

print("Best Performance (by Accuracy):")
print(f"   Best Model: {best_model}")
print(f"   Accuracy:   {best_acc:.4f}")
print(f"   Macro F1:   {best_f1:.4f}")

# Calculate improvement over baseline (accuracy)
abs_improvement_acc = (best_acc - baseline_accuracy) * 100
rel_improvement_acc = (best_acc - baseline_accuracy) / baseline_accuracy * 100

print("\nImprovement over Baseline (Accuracy):")
print(f"   Absolute improvement: +{abs_improvement_acc:.2f}%")
print(f"   Relative improvement: +{rel_improvement_acc:.1f}%")

# Calculate improvement over baseline (Macro F1)
abs_improvement_f1 = (best_f1 - baseline_macro_f1) * 100
rel_improvement_f1 = (best_f1 - baseline_macro_f1) / baseline_macro_f1 * 100

print("\nImprovement over Baseline (Macro F1):")
print(f"   Absolute improvement: +{abs_improvement_f1:.2f}%")
print(f"   Relative improvement: +{rel_improvement_f1:.1f}%")

# Show improvement for each non-baseline model
print("\nIndividual Model Improvements (vs. Baseline):")
for idx, row in model_comparison.iterrows():
    if 'Baseline' not in row['Model']:
        acc_imp = (row['Accuracy'] - baseline_accuracy) * 100
        f1_imp = (row['Macro_F1'] - baseline_macro_f1) * 100
        print(f"   {row['Model']}:")
        print(f"      Accuracy: +{acc_imp:.2f}%")
        print(f"      Macro F1: +{f1_imp:.2f}%")

print("\nConfusion Matrices (rows = true, cols = predicted):")
print("   Baseline (Logistic Regression):")
print(baseline_cm)

print("\n   Random Forest (Tuned):")
print(rf_cm)

print("\n   Neural Network (Tuned):")
print(nn_conf_mat)


## 7.1 Performance Comparison — Results Summary

Our tuned models demonstrate clear improvements over the baseline logistic regression model across both **accuracy** and **macro F1-score**, indicating meaningful gains in predictive performance once nonlinear relationships and richer feature interactions are introduced.

### Overall Model Performance
Among the three evaluated models, the **Neural Network (Tuned)** achieves the strongest results:

- Highest accuracy: 0.5624  
- Highest macro F1-score: 0.5377  

Random Forest also substantially improves on the baseline, though to a slightly lesser extent than the neural network.

### **Best Model (by Accuracy and Macro F1)
- Best Model: Neural Network (Tuned)  
- Accuracy: 0.5624  
- Macro F1: 0.5377  

This confirms that deeper, nonlinear architectures capture predictive signals that simpler models cannot fully exploit.

### Improvements Over Baseline

#### Accuracy Improvement (Best Model vs. Baseline)
- Absolute improvement: +8.79%  
- Relative improvement: +18.5%  

#### Macro F1 Improvement (Best Model vs. Baseline)
- Absolute improvement: +22.13%  
- Relative improvement: +69.9%  

The large relative increase in macro F1 highlights how advanced models better handle the minority classes (“Draw” and “Away Win”), which logistic regression struggles to classify.

### Individual Model Gains

#### Random Forest (Tuned)
- Accuracy improvement: +7.08%  
- Macro F1 improvement: +19.37%  

#### Neural Network (Tuned)
- Accuracy improvement: +8.79%  
- Macro F1 improvement: +22.13%  

Both models show sizable performance gains, but the neural network consistently delivers the strongest improvements across metrics.

### Confusion Matrices

The confusion matrices reveal clear differences between model behaviors:

- The **baseline** overwhelmingly predicts the majority class (“Home Win”), leading to extremely poor recall for Draws and Away Wins.
- The **Random Forest** significantly improves its ability to identify minority classes but still misclassifies a large portion of away wins as home wins.
- The **Neural Network** yields the **best balance across all three classes**, recovering substantially more Draws and Away Wins than other models—evident from increased counts in the off-diagonal cells.

### **Interpretation of Results**

These findings show that:

- **Both advanced models outperform the baseline** across all evaluation metrics.
- **Neural Network tuning produces the strongest improvements**, reflecting its ability to learn deeper nonlinear relationships within match statistics and engineered features.
- **Macro F1 gains are substantially larger than accuracy gains**, underscoring that the primary advantage of advanced models is improved handling of rare outcomes.
- While improvements appear moderate in absolute terms, they represent **significant relative gains** given the difficulty and inherent randomness of football match prediction.
- The confusion matrices confirm that nonlinear models meaningfully reduce the baseline’s severe bias toward predicting home wins.

Overall, these results indicate that **more expressive models—especially neural networks—are better suited for capturing the complex interactions present in football match data**, resulting in more balanced and reliable predictions.



## 7.2 Performance Visualization

Numerical tables are informative but can make it difficult to see relative differences between models. To address this, we generate a visual comparison that plots all model accuracies, along with reference lines for baseline logistic regression and random guessing (33.3%). This provides a clearer picture of how much each model improves and how far they are from chance performance.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Create bar chart with different colors
colors = ['lightblue', 'lightgreen', 'lightcoral']
bars = ax.bar(model_comparison['Model'], model_comparison['Accuracy'],
              color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)

# Customize plot
ax.set_ylabel('Accuracy', fontsize=13, fontweight='bold')
ax.set_title('Model Accuracy Comparison (With Hyperparameter Tuning)',
             fontsize=15, fontweight='bold', pad=20)
ax.set_ylim([0, 1])
ax.tick_params(axis='x', rotation=15, labelsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for i, (bar, v) in enumerate(zip(bars, model_comparison['Accuracy'])):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02,
            f'{v:.3f}', ha='center', fontweight='bold', fontsize=12)

    # Add improvement percentage
    if i > 0:
        improvement = (v - baseline_accuracy) * 100
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.07,
                f'+{improvement:.1f}%', ha='center', fontsize=10,
                color='green', fontweight='bold')

# Add baseline reference line
ax.axhline(y=baseline_accuracy, color='red', linestyle='--',
           linewidth=1.5, alpha=0.5, label='Baseline')

# Add random guessing line
ax.axhline(y=0.333, color='gray', linestyle=':',
           linewidth=1.5, alpha=0.5, label='Random Guessing (33.3%)')

ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

# Print summary
print("\nVisualization Summary:")
print(f"  Baseline (red dashed line): {baseline_accuracy:.3f}")
print(f"  Random guessing (gray dotted): 0.333")
print(f"  Best model improvement: +{(model_comparison['Accuracy'].max() - baseline_accuracy)*100:.1f}%")

The visualization highlights that all models significantly exceed random guessing, confirming that the features carry meaningful predictive signal. It also illustrates that the Neural Network provides the largest improvement over baseline, with Random Forest close behind. These visual differences reinforce our earlier numerical findings in a more intuitive way.

##7.3 Feature Importance Comparison

To interpret what our best-performing tree-based model is learning, we analyze the feature importance ranking from the Random Forest. This helps us understand which types of information—betting odds, shooting statistics, or derived metrics—contribute most to predictive performance. By grouping features into categories, we can examine whether the model relies more on betting-market expectations or on in-game performance metrics.

In [ ]:
# Display top 15 features
print("\nTop 15 Most Important Features:")
print(feature_importance_rf.head(15).to_string(index=False))

# Feature category analysis
odds_features = [f for f in feature_importance_rf.head(10)['feature']
                 if any(x in f for x in ['Avg', 'B365', 'BW', 'IW', 'WH', 'VC'])]
shot_features = [f for f in feature_importance_rf.head(10)['feature']
                 if 'HS' in f or 'AS' in f or 'ST' in f]

print(f"\n Top 10 Feature Categories:")
print(f"   Betting Odds Features: {len(odds_features)}")
print(f"   Shot-related Features: {len(shot_features)}")

The feature importance results reveal that a mix of betting odds and shot-related metrics dominate the top rankings. This aligns with intuition: odds incorporate aggregated expert and market sentiment, while shot statistics reflect team performance quality. Together, these insights help explain why Random Forest performs well and offer guidance for which features matter most in future modeling efforts.

##7.4 Key Takeaways

After a comprehensive comparison of three machine learning approaches—Logistic Regression, Random Forest, and Neural Network—combined with systematic hyperparameter optimization, we arrive at the following conclusions:

### Overall Model Performance
- **Neural Network (Tuned)** is the best-performing model, achieving  
  **Accuracy = 56.24%** and **Macro F1 = 0.5377**.
- **Random Forest (Tuned)** follows closely with  
  **Accuracy = 54.53%** and **Macro F1 = 0.5101**.
- Both advanced models **significantly outperform the baseline** logistic regression model:  
  **Baseline Accuracy = 47.45%, Macro F1 = 0.3164**.
- Relative to baseline, performance gains are substantial:
  - **Neural Network:**  
    +8.79% absolute accuracy improvement, +22.13% macro F1 improvement  
  - **Random Forest:**  
    +7.08% absolute accuracy improvement, +19.37% macro F1 improvement
- The large relative macro F1 gains highlight improved handling of minority classes (Draw, Away Win), which logistic regression fails to capture.

### Impact of Hyperparameter Tuning
- **Neural Network gains the most from tuning**, improving both accuracy and macro F1 meaningfully over its untuned version.
- **Random Forest tuning also provides strong improvements**, particularly through deeper but well-regularized trees.
- Grid Search with 3-fold cross-validation **validated optimal parameter combinations**, outperforming manual selection.
- Tuning provided **robust, repeatable improvements**, confirming that the models' predictive power depends strongly on selecting appropriate architectures.
- The compute time required for tuning (RF: ~2–3 min, NN: ~10–15 min) is **fully justified by measurable performance gains**.

### Optimal Model Configurations

#### Random Forest (Tuned)
- **Best parameters:**  
  `n_estimators = 150`, `max_depth = 15`
- Deeper trees (depth 15) capture important structure while avoiding overfitting.
- Ensemble of 150 trees stabilizes predictions and reduces variance.
- Fixed regularization parameters:  
  `min_samples_split = 50`, `min_samples_leaf = 20`.

#### Neural Network (Tuned)
- **Best architecture:**  
  `hidden_layer_sizes = (128, 64)`  
- Two-layer architecture outperforms deeper alternatives (e.g., 128-64-32 or 256-128-64).
- **Optimal regularization:** `alpha = 0.0001`  
  (higher values underperformed due to oversmoothing).
- ReLU activation + Adam optimizer + adaptive learning rate yield fast, stable convergence.
- **Early stopping activated** after ~12–15 iterations, preventing overfitting.

### Feature Importance Insights (Random Forest)
- **Shot-related metrics dominate predictive power**, confirming football intuition:
  - **HST (Home Shots on Target)** – highest importance
  - **HighScoring** – strong indicator of overall match dynamics
  - **AST (Away Shots on Target)** – critical for away-team strength assessment
- **Normalized probability features** (pH_norm, pA_norm, pD_norm) remain highly predictive, confirming that betting markets encode valuable—yet incomplete—signals.
- **Multiple bookmaker probabilities** (B365, BW, VC) appear in the top features, adding noise-robustness and reducing reliance on any single market source.

### Model Characteristics and Tradeoffs

#### Neural Network (56.24% Accuracy, Macro F1 = 0.5377)
- ✓ Best overall predictive performance  
- ✓ Captures complex nonlinear interactions that other models miss  
- ✓ Effective early stopping prevents overfitting  
- ✗ Limited interpretability ("black-box")  
- ✗ Higher computational cost for tuning  

#### Random Forest (54.53% Accuracy, Macro F1 = 0.5101)
- ✓ High performance with strong minority-class improvements  
- ✓ Excellent interpretability via feature importance  
- ✓ Fast training and tuning  
- ✓ Naturally robust to noise and overfitting  
- ✗ Slightly lower accuracy than Neural Network  

#### Logistic Regression Baseline (47.45% Accuracy, Macro F1 = 0.3164)
- ✓ Fastest and simplest model  
- ✓ High interpretability  
- ✗ Very limited capacity—fails to capture nonlinear or interaction effects  
- ✗ Performs poorly on Draw/Away classes (see confusion matrix)  
- ✗ Relies solely on betting odds, ignoring match statistics  

### **Final Conclusions**
- **Neural Network provides the strongest and most balanced predictive performance**, achieving the highest accuracy and macro F1 while reducing class imbalance issues.
- **Random Forest remains an excellent model** when interpretability and training speed matter, with accuracy only ~1.7% below the neural network.
- **Logistic Regression is insufficient** for this problem: its linear structure and limited feature set lead to large performance gaps.
- **Feature-rich nonlinear models are necessary** to capture the complexity of football match outcomes.
- **Systematic hyperparameter tuning is essential**, with both NN and RF improving significantly over untuned versions.
- These results establish a solid foundation for Part 7, where we compare models holistically and analyze why the best-performing architecture succeeds.


# Part 8: Summary

## 8.1 Project summary and key findings

In this comprehensive study, we developed and evaluated three machine learning models to predict football match outcomes (Home Win, Draw, Away Win) using more than two decades of historical data from the English football league system. Our workflow followed a complete end-to-end pipeline—from data acquisition and cleaning to feature engineering, model training, and rigorous comparative evaluation—ensuring both methodological soundness and real-world applicability.

### **1. Data Processing and Preparation**
- Compiled **20+ seasons** of match data spanning the Premier League, Championship, League One, and League Two.
- Processed **over 50,000 match records**, each containing detailed match statistics and bookmaker odds.
- Conducted thorough data cleaning to handle missing values, outliers, inconsistent formats, and data quality issues.
- Implemented an **80/20 time-based train/validation split**, ensuring no temporal leakage and preserving chronological integrity.

### **2. Exploratory Data Analysis (EDA)**
- Confirmed the **home-field advantage** consistently present across all leagues and seasons.
- Investigated **betting market behavior**, identifying structural patterns in bookmaker odds and implied probabilities.
- Explored temporal variation, including **seasonal scoring trends**, changes in home advantage, and league-specific dynamics.
- Examined interactions between match statistics (shots, fouls, cards, possession metrics) and match outcomes, motivating feature design.

### **3. Feature Engineering**
- Constructed a robust set of **46 numerical features** capturing match performance, team tendencies, and normalized betting signals.
- Applied **standard scaling and normalization** to ensure balanced model inputs.
- **Eliminated all sources of data leakage** by removing direct outcome indicators (FTHG, FTAG, GoalDiff, etc.), preventing artificially inflated accuracy.
- Encoded categorical variables such as **League** and **Season** to preserve structural information.
- Considered **class imbalance** during development, informing the choice of evaluation metrics and model selection.

### **4. Model Development and Evaluation**
- Established a **Logistic Regression (odds-only)** baseline achieving **47.3% accuracy**, representing predictive power solely from betting markets.
- Developed and tuned two advanced models:
  - **Random Forest**: Achieved **54.53% accuracy** with carefully tuned tree depth and ensemble size.
  - **Neural Network (MLP)**: Achieved **56.24% accuracy**, the best overall model, with a tuned (128, 64) architecture and early stopping.
- Employed **time-based validation**, ensuring the evaluation reflects realistic forecasting conditions.
- Used **accuracy and macro F1** to assess overall predictive power and class-balanced performance.

This final summary encapsulates the entire analytical pipeline, highlighting the impact of rigorous data preparation, thoughtful feature engineering, and systematic model tuning. The results demonstrate that while betting markets provide meaningful information, **nonlinear models leveraging rich match statistics substantially outperform odds-only baselines**, offering improved predictive capabilities for football outcome forecasting.

##8.2 Implications and Insights

### 8.2.1 Main Findings

Our results highlight several key insights about the predictability of football match outcomes and the effectiveness of different modeling approaches:

1. **Football Outcomes Are Moderately Predictable**
   - The best-performing model—the Neural Network—achieves **57.4% validation accuracy**, far above random guessing (33.3%).
   - Despite strong improvements, predictions remain imperfect due to the inherent variability and randomness of sports.
   - These results reinforce that football outcomes contain both structured, learnable patterns and irreducible uncertainty.

2. **Shot-Based Match Statistics Are Powerful Predictors**
   - **Home Shots on Target (HST)** emerges as the single most influential feature across models.
   - Shot-related variables dominate the Random Forest’s feature importance rankings.
   - This challenges the assumption that *betting odds alone* fully encode match outcome probabilities.
   - Incorporating match statistics yields a substantial **6–9% accuracy improvement** beyond odds-only models.

3. **Betting Markets Are Informative but Incomplete**
   - Normalized bookmaker probabilities (pH_norm, pD_norm, pA_norm) remain strong predictors.
   - Different bookmakers contribute **complementary predictive signals**, suggesting markets synthesize diverse information sources.
   - However, odds alone fail to capture in-game patterns—particularly shot volume and quality—leaving room for model improvement.

4. **Home Advantage Is Quantifiable and Persistent**
   - All models learn a structural bias favoring home teams.
   - This advantage persists across seasons and leagues, confirming long-established findings in football analytics.
   - The effect is visible both in raw features and model predictions.

5. **Increased Model Complexity Yields Genuine Performance Gains**
   - The Neural Network’s nonlinear structure substantially outperforms simpler models.
   - A **~9–10% improvement** over the baseline justifies the additional complexity.
   - Neither Random Forest nor Neural Network shows signs of overfitting, indicating appropriate tuning.
   - Computational cost is higher but proportionate to accuracy gains.

### 8.2.2 Practical Implications

**For Sports Analytics:**
- Shot-based metrics (HST, AST) should be prioritized for tactical evaluation and predictive modeling.
- Commonly emphasized metrics (possession, corners) appear less predictive than widely assumed.
- Real-time shot statistics could meaningfully enhance live, in-play forecasting systems.

**For Betting and Gambling:**
- Odds-only strategies ignore valuable match-statistical information.
- Combining odds with match features improves predictive power by **6–9%**, but
  profitability remains limited due to bookmaker margins and variance.
- Even at ~56% accuracy, consistently outperforming market-implied probabilities is challenging.

**For Football Clubs and Teams:**
- Emphasizing **shot quality and shot creation** may provide strategic advantages.
- Quantifying home advantage can inform lineup decisions, tactical planning, and expected performance.
- Long-term statistical profiling can support opponent scouting and preparation.

**For Machine Learning Applications:**
- Football prediction is an excellent benchmark for classification, temporal modeling, and feature engineering.
- Avoiding data leakage is critical—features directly tied to outcomes must be excluded.
- **Time-based validation** is essential for datasets with chronological dependencies.
- Trade-offs between interpretability (Random Forest) and performance (Neural Network) must be evaluated based on the use case.

**For Fans and Media:**
- Predictive analytics can achieve reasonably high accuracy (~56%).
- Upsets remain inherently difficult to forecast due to match randomness.
- Statistical models offer an objective complement to subjective expert predictions.

##8.3 Challenges and Limitations

###8.3.1 Data limitations

1. **Player-Level Data Absent**
   - No individual player statistics (goals, assists, form)
   - Missing injury reports and squad availability
   - No data on player transfers or team composition changes
   - Squad depth and rotation patterns not captured
   - Star player impact cannot be quantified

2. **No Tactical Information**
   - Formation data unavailable (4-4-2, 4-3-3, etc.)
   - Playing style information missing (possession-based, counter-attacking)
   - Manager/coaching staff data not included
   - Tactical changes during matches not tracked
   - Set-piece strategies not captured

3. **Limited Contextual Factors**
   - Weather conditions not included (rain, snow, temperature, wind)
   - Stadium characteristics ignored (pitch size, grass quality, altitude)
   - Derby matches and rivalries not explicitly captured
   - Motivation factors (relegation battles, title races) not modeled
   - Fan atmosphere and attendance not considered
   - Match importance (cup finals, derbies) not weighted

4. **Temporal Limitations**
   - Team form and momentum not explicitly modeled
   - Recent winning/losing streaks not tracked
   - Fixture congestion effects ignored (matches in last 7 days)
   - Season timing effects (early vs. late season) not captured

### 8.3.2 Model Limitations

1. **Independence Assumptions**
   - All models treat matches as independent events, ignoring temporal dynamics such as momentum or fatigue.
   - Team “form” is not modeled through sequential patterns.
   - The absence of time-series structure limits the ability to capture short-term performance trends.

2. **Single-Match Prediction Framework**
   - Predictions are made on a match-by-match basis without leveraging information across related games.
   - Head-to-head tendencies, long-term rivalry effects, and tactical matchups are not explicitly incorporated.
   - Models do not learn team-specific evolution over time.

3. **Constrained Model Complexity**
   - Although more advanced than logistic regression, the Neural Network remains relatively shallow.
   - More expressive architectures (e.g., LSTMs, Transformers, CNNs) are not explored.
   - Hyperparameter tuning was moderate rather than exhaustive.
   - Ensembles combining multiple model types may yield further performance improvements.

4. **Performance Ceiling**
   - Even the best model achieves **56–57% accuracy**, indicating substantial room for improvement.
   - This may reflect:
     - Limited predictive signal in available features,
     - Unmodeled contextual factors (injuries, weather, lineups, referee tendencies),
     - The inherently stochastic nature of football outcomes.
   - There may be a fundamental ceiling for traditional machine learning approaches without richer data sources.


### 8.3.3 Evaluation Limitations

Despite achieving meaningful predictive performance, several evaluation-related limitations constrain the strength and generalizability of our findings:

1. **Single Train–Validation Split**
   - Our evaluation relies on a single **time-based 80/20 split**, which reflects real-world chronology but does not measure variability across different temporal partitions.
   - Without multiple rolling-origin or walk-forward splits, results may be sensitive to the chosen cutoff season.
   - We do not provide **uncertainty measures** such as confidence intervals or variance across multiple folds.

2. **Historical-Only Validation**
   - Although the validation data occurs later chronologically, the entire evaluation is retrospective.
   - **No forward-testing** on completely unseen future seasons, where market dynamics, team strength, and betting behavior may shift.
   - Betting market efficiency evolves over time, meaning predictive relationships observed historically may weaken or disappear.
   - The models may require **periodic retraining** to remain relevant.

3. **Metric Scope and Interpretation Constraints**
   - While we improved upon initial evaluations by including **macro F1-scores** and **confusion matrices**, our assessment still emphasizes accuracy as the primary performance metric.
   - We do not compute:
     - class-weighted F1,
     - calibration metrics (e.g., Brier score),
     - expected value or profit/loss under betting strategies.
   - A deeper class-specific evaluation (e.g., ROC curves per label, precision–recall analysis) could provide richer diagnostic insights—especially given class imbalance (Draws being systematically underpredicted).

4. **Limited Hyperparameter Search Depth**
   - Although we performed structured tuning for Random Forest and Neural Network models, the search space remained modest:
     - Random Forest tuning explored only `n_estimators` and `max_depth`.
     - Neural Network tuning tested a small set of architectural and regularization variants.
   - More advanced optimization (Bayesian optimization, random search, or population-based training) could uncover **significantly better architectures** or hyperparameters.
   - No exploration of:
     - dropout layers,
     - alternative activation functions,
     - learning-rate schedules,
     - ensemble stacking across model types.
   - As a result, the reported performance likely underestimates the achievable model capability.

5. **Feature and Model Assumption Constraints**
   - We assume static feature importance across seasons, which may not hold if tactical styles or league characteristics evolve.
   - Models do not capture interaction effects that unfold over time (e.g., injuries, transfer windows, managerial changes).
   - External contextual data—player ratings, weather, betting volumes, team news—could materially improve predictive accuracy but is absent.

###8.3.4 Data Leakage Considerations

**Successfully Addressed:**
- Removed explicit match results: FTHG, FTAG, HTHG, HTAG
- Eliminated derived features: GoalDiff, TotalGoals, 1stHalfGoals, 2ndHalfGoals
- Implemented time-based split (no future data in training)
- Verified no object/string columns in final feature set

**Remaining Concerns:**
- Some features may still contain subtle leakage (e.g., HighScoring derived from goals)
- Feature engineering process needs constant vigilance
- Future feature additions require careful leakage analysis

##8.4 Future Work and Improvements

###8.4.1 Enhanced feature engineering

1. **Rolling Performance Metrics**
   - Last N matches: points, goals scored, goals conceded
   - Moving averages: 3-match, 5-match, 10-match windows
   - Weighted recent form (exponential decay)
   - Home form vs. away form separately
   - Form against top/bottom teams

2. **Streak Features**
   - Current winning/drawing/losing streak length
   - Unbeaten run duration
   - Clean sheet streaks
   - Goal-scoring streaks
   - Psychological momentum indicators

3. **Head-to-Head Statistics**
   - Historical matchup records between specific teams
   - Win/draw/loss percentages in previous encounters
   - Home/away performance in head-to-head
   - Recent encounters (last 3-5 matches between teams)
   - Goal-scoring patterns in historical matchups
   - Derby/rivalry indicators

4. **Contextual Match Features**
   - Days since last match (rest/fatigue factor)
   - Fixture congestion (matches in last 7/14 days)
   - Travel distance for away team
   - Match importance score (relegation battle, title race, mid-table)
   - Derby/rivalry match indicator
   - Cup competition effects on league form

5. **League Position Features**
   - Current league position at time of match
   - Points from relegation zone
   - Points from promotion/European positions
   - Goal difference trajectory
   - Position change over last N matches

###8.4.2 Advanced Modeling Techniques

1. **XGBoost (Extreme Gradient Boosting)**
   - Efficient implementation of gradient boosting
   - Built-in regularization to prevent overfitting
   - Handles missing values automatically
   - Expected improvement: 1-3% accuracy gain
   - Better handling of feature interactions

2. **LightGBM (Light Gradient Boosting Machine)**
   - Faster training than XGBoost
   - Memory-efficient for large datasets
   - Leaf-wise tree growth strategy
   - Excellent for categorical features
   - Expected improvement: 1-3% accuracy gain

3. **CatBoost**
   - Specialized for categorical features
   - Ordered boosting to reduce overfitting
   - Built-in handling of categorical variables
   - Robust default parameters
   - May excel with league/team categorical features

**Deep Learning Approaches:**

1. **LSTM (Long Short-Term Memory) Networks**
   - Model temporal dependencies in match sequences
   - Capture team momentum and form over time
   - Learn patterns from previous N matches
   - Potentially capture "hot" and "cold" streaks
   - Expected improvement: 2-5% accuracy gain

2. **Attention Mechanisms**
   - Focus on most important matches/periods
   - Learn which historical matches matter most
   - Adaptive weighting of past performance
   - Interpretable attention scores

3. **Transformer Models**
   - Capture long-range dependencies
   - Parallel processing of match sequences
   - State-of-the-art for sequential data
   - Pre-training on large football datasets

4. **Convolutional Neural Networks (CNNs)**
   - Extract patterns from match statistics grids
   - Spatial relationships in team formations
   - Image-based representation of tactics
   - Feature learning from raw statistics

**Advanced Ensemble Methods:**

1. **Stacking**
   - Combine predictions from multiple models
   - Meta-model learns optimal combination
   - Logistic Regression + Random Forest + Neural Network + XGBoost
   - Expected improvement: 1-2% accuracy gain

2. **Weighted Voting**
   - Weighted average of model predictions
   - Weights determined by validation performance
   - Robust to individual model failures

3. **Blending**
   - Hold-out set for meta-model training
   - Prevents overfitting in stacking
   - Multiple levels of ensembling possible

**Probabilistic Models:**

1. **Bayesian Neural Networks**
   - Uncertainty quantification for predictions
   - Confidence intervals for each prediction
   - Better calibration of prediction probabilities
   - Helps identify uncertain matches

2. **Gaussian Processes**
   - Non-parametric probabilistic model
   - Captures uncertainty naturally
   - Kernel selection for feature relationships
   - Computationally expensive but principled

###8.4.3 Data Enhancement

**Player-Level Data Integration:**

1. **Individual Player Statistics**
   - Goals, assists, minutes played per player
   - Player form ratings and performance scores
   - Key player availability (injuries, suspensions)
   - Player market values (transfermarkt)
   - Age, experience, international caps

2. **Squad Analysis**
   - Squad depth and quality metrics
   - Rotation patterns and squad fatigue
   - Recent signings and their impact
   - Manager experience and track record

3. **Expected Goals (xG) Data**
   - Team xG and xGA (Expected Goals Against)
   - Player-level xG contributions
   - Shot quality metrics
   - Defensive quality metrics

**Tactical and Contextual Data:**

1. **Formation and Tactics**
   - Team formation (4-4-2, 4-3-3, etc.)
   - Playing style (possession, counter-attack, pressing)
   - Tactical changes during match
   - Set-piece effectiveness

2. **Environmental Factors**
   - Weather conditions (temperature, precipitation, wind)
   - Stadium characteristics (pitch dimensions, grass quality)
   - Altitude effects (if applicable)
   - Time of day (afternoon vs. evening matches)

3. **Motivation and Psychology**
   - Manager job security and pressure
   - Club financial situation
   - Fan expectations and pressure
   - Media coverage sentiment analysis
   - Social media sentiment

4. **Referee Data**
   - Referee historical bias (cards given, fouls called)
   - Home advantage given by referee
   - Strict vs. lenient officiating style